# Cross-Image GLOT — persistent Drive storage with local runtime copies

Google Drive is the persistent source of truth. Colab's `/content` directory is the local working copy.

Normal runtime behavior:

1. Mount Google Drive.
2. Keep `images.zip`, the CSV manifests, and all DINOv2 feature shards persistently in Drive.
3. Copy the dataset archive and cached feature shards from Drive to `/content`.
4. Extract images locally and train from local files for faster I/O.
5. Instantiate DINOv2 only when a required feature shard is missing.
6. Save each newly generated shard locally and immediately mirror it to Drive.

After the feature cache is complete, later runtimes copy it from Drive and never run DINOv2 again.


In [139]:
from __future__ import annotations

from collections import OrderedDict
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import json
import math
import os
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from google.colab import drive

drive.mount("/content/drive")

# Persistent source of truth.
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CrossImageGLOT")
DRIVE_DATA_DIR = DRIVE_PROJECT_ROOT / "data"
DRIVE_CACHE_DIR = DRIVE_PROJECT_ROOT / "features" / "dinov2_vits14_224"

# Fast local working copy. This disappears when the Colab runtime resets.
LOCAL_PROJECT_ROOT = Path("/content/CrossImageGLOT")
LOCAL_DATA_DIR = LOCAL_PROJECT_ROOT / "data"
LOCAL_IMAGE_DIR = LOCAL_DATA_DIR / "images" / "images"
LOCAL_CACHE_DIR = LOCAL_PROJECT_ROOT / "features" / "dinov2_vits14_224"

for directory in (
    DRIVE_DATA_DIR,
    DRIVE_CACHE_DIR,
    LOCAL_DATA_DIR,
    LOCAL_CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Persistent data:", DRIVE_DATA_DIR)
print("Persistent embeddings:", DRIVE_CACHE_DIR)
print("Local data:", LOCAL_DATA_DIR)
print("Local embeddings:", LOCAL_CACHE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent data: /content/drive/MyDrive/CrossImageGLOT/data
Persistent embeddings: /content/drive/MyDrive/CrossImageGLOT/features/dinov2_vits14_224
Local data: /content/CrossImageGLOT/data
Local embeddings: /content/CrossImageGLOT/features/dinov2_vits14_224


## Optional one-time migration from an existing runtime

Run this once if the earlier notebook already created files under `/content/data` or `/content/features/dinov2_vits14_224`. It preserves those files in Google Drive before the runtime is reset.

It is safe to run when those paths do not exist.


In [140]:
LEGACY_DATA_DIR = Path("/content/data")
LEGACY_CACHE_DIR = Path("/content/features/dinov2_vits14_224")

for file_name in ("train.csv", "val.csv", "test.csv", "images.zip"):
    source = LEGACY_DATA_DIR / file_name
    destination = DRIVE_DATA_DIR / file_name
    if source.exists() and not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        print(f"Migrating {source} -> {destination}")
        shutil.copy2(source, destination)

if LEGACY_CACHE_DIR.exists():
    print(f"Migrating existing feature shards -> {DRIVE_CACHE_DIR}")
    shutil.copytree(LEGACY_CACHE_DIR, DRIVE_CACHE_DIR, dirs_exist_ok=True)
    print("Legacy feature-cache migration completed.")
else:
    print("No legacy local feature cache was found.")


No legacy local feature cache was found.


## Persistent dataset archive and local extraction

Drive keeps the three CSV manifests and `images.zip` permanently. On each fresh runtime, these four files are copied to `/content`, and the ZIP is extracted locally.

Keeping the ZIP in Drive is preferable to reading 60,000 separate JPEG files through the Drive mount on every run.


In [141]:
from zipfile import ZipFile

SHARED_FOLDER_ID = "1iDi3zKexa6f3FGTLUVD5SgrnaAa3RlUx"
DATASET_FILES = ("train.csv", "val.csv", "test.csv", "images.zip")


def copy_file_if_needed(source: Path, destination: Path) -> bool:
    """Copy a file only when the destination is absent or has a different size."""
    if not source.exists():
        raise FileNotFoundError(source)

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size == source.stat().st_size:
        return False

    temporary = destination.with_suffix(destination.suffix + ".tmp")
    if temporary.exists():
        temporary.unlink()

    shutil.copy2(source, temporary)
    os.replace(temporary, destination)
    return True


# Download missing source files once into persistent Drive storage.
missing_in_drive = {
    file_name
    for file_name in DATASET_FILES
    if not (DRIVE_DATA_DIR / file_name).exists()
}

if missing_in_drive:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    auth.authenticate_user()
    drive_service = build("drive", "v3")

    result = drive_service.files().list(
        q=f"'{SHARED_FOLDER_ID}' in parents and trashed = false",
        fields="files(id, name)",
    ).execute()

    available = {item["name"]: item["id"] for item in result.get("files", [])}
    unavailable = missing_in_drive - set(available)
    if unavailable:
        raise FileNotFoundError(
            f"Files missing from the shared folder: {sorted(unavailable)}"
        )

    for file_name in sorted(missing_in_drive):
        destination = DRIVE_DATA_DIR / file_name
        print(f"Persistently downloading {file_name} -> {destination}")

        request = drive_service.files().get_media(fileId=available[file_name])
        temporary = destination.with_suffix(destination.suffix + ".tmp")
        with temporary.open("wb") as output_file:
            downloader = MediaIoBaseDownload(output_file, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        os.replace(temporary, destination)
else:
    print("Persistent dataset archive already exists in Drive.")


# Copy the persistent archive/manifests to the local runtime.
for file_name in DATASET_FILES:
    copied = copy_file_if_needed(
        DRIVE_DATA_DIR / file_name,
        LOCAL_DATA_DIR / file_name,
    )
    print(f"{file_name}: {'copied to local runtime' if copied else 'local copy already valid'}")


# Extract locally. The marker belongs to the temporary runtime, not Drive.
LOCAL_EXTRACTION_MARKER = LOCAL_DATA_DIR / ".miniimagenet_extracted"
local_image_count = (
    sum(1 for _ in LOCAL_IMAGE_DIR.glob("*.jpg"))
    if LOCAL_IMAGE_DIR.exists()
    else 0
)

if not LOCAL_EXTRACTION_MARKER.exists() or local_image_count != 60_000:
    extraction_root = LOCAL_DATA_DIR / "images"
    if extraction_root.exists():
        shutil.rmtree(extraction_root)
    extraction_root.mkdir(parents=True, exist_ok=True)

    print("Extracting miniImageNet into the local runtime...")
    with ZipFile(LOCAL_DATA_DIR / "images.zip", "r") as archive:
        archive.extractall(extraction_root)

    local_image_count = sum(1 for _ in LOCAL_IMAGE_DIR.glob("*.jpg"))
    if local_image_count != 60_000:
        raise RuntimeError(
            f"Expected 60,000 local images, found {local_image_count:,}."
        )

    LOCAL_EXTRACTION_MARKER.write_text("60000\n", encoding="utf-8")
    print("Local extraction completed.")
else:
    print("Local miniImageNet extraction already exists.")

for split in ("train", "val", "test"):
    if not (LOCAL_DATA_DIR / f"{split}.csv").exists():
        raise FileNotFoundError(LOCAL_DATA_DIR / f"{split}.csv")


Persistent dataset archive already exists in Drive.
train.csv: local copy already valid
val.csv: local copy already valid
test.csv: local copy already valid
images.zip: local copy already valid
Local miniImageNet extraction already exists.


## Raw-image dataset

This dataset is used only when a missing cache must be built. Once all cache splits exist, training uses `MiniImageNetFeatureDataset` below and does not open JPEG files.


In [142]:
class MiniImageNetImageDataset(Dataset):
    EXPECTED = {
        "train": (38_400, 64),
        "val": (9_600, 16),
        "test": (12_000, 20),
    }

    def __init__(
        self,
        data_dir: str | Path,
        image_dir: str | Path,
        split: str,
        transform: Callable,
    ) -> None:
        if split not in self.EXPECTED:
            raise ValueError(f"Unknown split: {split}")

        self.data_dir = Path(data_dir)
        self.image_dir = Path(image_dir)
        self.split = split
        self.transform = transform

        self.frame = pd.read_csv(self.data_dir / f"{split}.csv")
        if set(self.frame.columns) < {"filename", "label"}:
            raise ValueError("CSV must contain filename and label columns.")

        self.frame = self.frame[["filename", "label"]].copy()
        self.frame["filename"] = self.frame["filename"].astype(str)
        self.frame["label"] = self.frame["label"].astype(str)

        expected_images, expected_classes = self.EXPECTED[split]
        if len(self.frame) != expected_images:
            raise ValueError(
                f"{split}: expected {expected_images:,} rows, found {len(self.frame):,}."
            )
        if self.frame["label"].nunique() != expected_classes:
            raise ValueError(
                f"{split}: expected {expected_classes} classes, "
                f"found {self.frame['label'].nunique()}."
            )

        counts = self.frame.groupby("label").size()
        if not counts.eq(600).all():
            raise ValueError(f"{split}: every class must contain 600 images.")

        self.class_ids = sorted(self.frame["label"].unique().tolist())
        self.class_to_indices = {
            class_id: self.frame.index[self.frame["label"] == class_id].tolist()
            for class_id in self.class_ids
        }

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict:
        row = self.frame.iloc[index]
        image_path = self.image_dir / row["filename"]

        with Image.open(image_path) as image:
            image = self.transform(image.convert("RGB"))

        return {
            "image": image,
            "dataset_index": index,
            "filename": row["filename"],
            "class_id": row["label"],
        }

    def indices_for_class(self, class_id: str) -> list[int]:
        return self.class_to_indices[class_id]


def validate_class_disjointness(data_dir: str | Path) -> None:
    data_dir = Path(data_dir)
    classes = {
        split: set(pd.read_csv(data_dir / f"{split}.csv")["label"].astype(str))
        for split in ("train", "val", "test")
    }

    assert classes["train"].isdisjoint(classes["val"])
    assert classes["train"].isdisjoint(classes["test"])
    assert classes["val"].isdisjoint(classes["test"])
    print("The train, validation and test classes are disjoint.")


def build_dinov2_transform(image_size: int = 224) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize(
                (image_size, image_size),
                interpolation=InterpolationMode.BICUBIC,
                antialias=True,
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        ]
    )


validate_class_disjointness(LOCAL_DATA_DIR)


The train, validation and test classes are disjoint.


## Frozen DINOv2 extractor

This class is instantiated only when cache creation is required.


In [143]:
@dataclass
class DINOv2Features:
    cls: torch.Tensor
    patches: torch.Tensor
    grid_size: tuple[int, int]


class DINOv2FeatureExtractor(nn.Module):
    def __init__(
        self,
        model_name: str = "dinov2_vits14",
        image_size: int = 224,
        device: str | torch.device | None = None,
        output_dtype: torch.dtype = torch.float16,
    ) -> None:
        super().__init__()

        self.model_name = model_name
        self.image_size = image_size
        self.patch_size = 14
        self.embedding_dim = 384
        self.grid_size = (image_size // self.patch_size,) * 2
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.output_dtype = output_dtype

        if image_size % self.patch_size:
            raise ValueError("image_size must be divisible by 14.")

        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.backbone = torch.hub.load(
            "facebookresearch/dinov2",
            model_name,
            pretrained=True,
            trust_repo=True,
        ).to(self.device)

        self.backbone.requires_grad_(False)
        self.backbone.eval()

    def train(self, mode: bool = True):
        super().train(False)
        self.backbone.eval()
        return self

    @torch.inference_mode()
    def forward(
        self,
        images: torch.Tensor,
        extraction_batch_size: int = 32,
        return_cpu: bool = True,
    ) -> DINOv2Features:
        if images.ndim < 4 or images.shape[-3] != 3:
            raise ValueError(f"Expected [..., 3, H, W], received {images.shape}.")
        if tuple(images.shape[-2:]) != (self.image_size, self.image_size):
            raise ValueError(
                f"Expected {self.image_size}x{self.image_size}, "
                f"received {tuple(images.shape[-2:])}."
            )

        leading_shape = images.shape[:-3]
        flat = images.reshape(-1, 3, self.image_size, self.image_size)
        destination = torch.device("cpu") if return_cpu else self.device

        cls_chunks, patch_chunks = [], []
        for start in range(0, len(flat), extraction_batch_size):
            batch = flat[start : start + extraction_batch_size].to(
                self.device, non_blocking=True
            )

            autocast = (
                torch.autocast("cuda", dtype=torch.float16)
                if self.device.type == "cuda"
                else nullcontext()
            )
            with autocast:
                output = self.backbone.forward_features(batch)

            cls_chunks.append(
                output["x_norm_clstoken"].to(destination, self.output_dtype)
            )
            patch_chunks.append(
                output["x_norm_patchtokens"].to(destination, self.output_dtype)
            )

        cls = torch.cat(cls_chunks).reshape(*leading_shape, self.embedding_dim)
        patches = torch.cat(patch_chunks).reshape(
            *leading_shape, self.num_patches, self.embedding_dim
        )

        return DINOv2Features(cls=cls, patches=patches, grid_size=self.grid_size)


## Sharded cache: local computation, persistent Drive mirror

Feature shards are used from the local runtime for faster training. Every newly generated shard is copied to Drive immediately. On later runtimes, the persistent Drive cache is copied back to `/content` before any missing-cache check is performed.


In [144]:
def _atomic_torch_save(value: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)


def _atomic_csv_save(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def copy_tree_update(source_root: Path, destination_root: Path) -> int:
    """Copy missing or size-mismatched files while preserving the directory tree."""
    if not source_root.exists():
        return 0

    copied_files = 0
    for source in source_root.rglob("*"):
        if not source.is_file() or source.name.endswith(".tmp"):
            continue

        relative = source.relative_to(source_root)
        destination = destination_root / relative
        if copy_file_if_needed(source, destination):
            copied_files += 1

    return copied_files


def split_cache_complete(
    cache_dir: str | Path,
    split: str,
    expected_images: int,
) -> bool:
    cache_dir = Path(cache_dir)
    split_dir = cache_dir / split
    index_path = split_dir / "index.csv"
    summary_path = split_dir / "summary.json"

    if not (cache_dir / "metadata.json").exists():
        return False
    if not index_path.exists() or not summary_path.exists():
        return False

    try:
        index = pd.read_csv(index_path)
    except Exception:
        return False

    if len(index) != expected_images:
        return False
    if index["dataset_index"].nunique() != expected_images:
        return False
    if set(index["split"].astype(str)) != {split}:
        return False

    return all(
        (split_dir / shard_name).exists()
        for shard_name in index["shard_name"].unique()
    )


class DINOv2FeatureCacheBuilder:
    FORMAT_VERSION = 1

    def __init__(
        self,
        extractor: DINOv2FeatureExtractor,
        local_cache_dir: str | Path,
        persistent_cache_dir: str | Path,
        images_per_shard: int = 256,
        extraction_batch_size: int = 32,
        num_workers: int = 2,
    ) -> None:
        self.extractor = extractor
        self.local_cache_dir = Path(local_cache_dir)
        self.persistent_cache_dir = Path(persistent_cache_dir)
        self.images_per_shard = images_per_shard
        self.extraction_batch_size = extraction_batch_size
        self.num_workers = num_workers

        self.local_cache_dir.mkdir(parents=True, exist_ok=True)
        self.persistent_cache_dir.mkdir(parents=True, exist_ok=True)

        metadata = {
            "format_version": self.FORMAT_VERSION,
            "model_name": extractor.model_name,
            "image_size": extractor.image_size,
            "patch_size": extractor.patch_size,
            "grid_size": list(extractor.grid_size),
            "num_patches": extractor.num_patches,
            "embedding_dim": extractor.embedding_dim,
            "dtype": str(extractor.output_dtype).replace("torch.", ""),
            "images_per_shard": images_per_shard,
            "preprocessing_id": "bicubic_resize_224_imagenet_normalization_v1",
        }

        metadata_path = self.local_cache_dir / "metadata.json"
        if metadata_path.exists():
            existing = json.loads(metadata_path.read_text(encoding="utf-8"))
            if existing != metadata:
                raise ValueError(
                    "Existing local cache metadata is incompatible with this extractor."
                )
        else:
            metadata_path.write_text(
                json.dumps(metadata, indent=2, sort_keys=True),
                encoding="utf-8",
            )

        self._persist_file(metadata_path)

    def _persist_file(self, local_path: Path) -> None:
        relative = local_path.relative_to(self.local_cache_dir)
        persistent_path = self.persistent_cache_dir / relative
        copy_file_if_needed(local_path, persistent_path)

    def build(self, dataset: MiniImageNetImageDataset) -> pd.DataFrame:
        split_dir = self.local_cache_dir / dataset.split
        persistent_split_dir = self.persistent_cache_dir / dataset.split
        split_dir.mkdir(parents=True, exist_ok=True)
        persistent_split_dir.mkdir(parents=True, exist_ok=True)
        index_path = split_dir / "index.csv"

        if split_cache_complete(
            self.local_cache_dir,
            dataset.split,
            len(dataset),
        ):
            print(f"{dataset.split}: local cache already complete; skipping DINOv2.")
            copy_tree_update(split_dir, persistent_split_dir)
            self._persist_file(self.local_cache_dir / "metadata.json")
            return pd.read_csv(index_path)

        loader = DataLoader(
            dataset,
            batch_size=self.images_per_shard,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=self.extractor.device.type == "cuda",
            persistent_workers=self.num_workers > 0,
        )

        records = []
        for shard_id, batch in enumerate(loader):
            shard_name = f"shard_{shard_id:05d}.pt"
            shard_path = split_dir / shard_name
            indices = batch["dataset_index"].long()
            expected = torch.arange(
                shard_id * self.images_per_shard,
                min((shard_id + 1) * self.images_per_shard, len(dataset)),
            )

            if not torch.equal(indices, expected):
                raise RuntimeError("Raw dataset order changed unexpectedly.")

            if shard_path.exists():
                shard = torch.load(
                    shard_path,
                    map_location="cpu",
                    weights_only=True,
                )
                valid = (
                    torch.equal(shard["dataset_indices"].long(), expected)
                    and tuple(shard["cls"].shape)
                    == (len(expected), self.extractor.embedding_dim)
                    and tuple(shard["patches"].shape)
                    == (
                        len(expected),
                        self.extractor.num_patches,
                        self.extractor.embedding_dim,
                    )
                )
                if not valid:
                    raise RuntimeError(f"Incompatible partial shard: {shard_path}")
                status = "reused"
            else:
                features = self.extractor(
                    batch["image"],
                    extraction_batch_size=self.extraction_batch_size,
                    return_cpu=True,
                )
                _atomic_torch_save(
                    {
                        "dataset_indices": indices.cpu().contiguous(),
                        "cls": features.cls.cpu().contiguous(),
                        "patches": features.patches.cpu().contiguous(),
                    },
                    shard_path,
                )
                status = "created"

            # Persist every completed shard immediately.
            self._persist_file(shard_path)

            for offset, dataset_index in enumerate(indices.tolist()):
                records.append(
                    {
                        "dataset_index": dataset_index,
                        "split": dataset.split,
                        "filename": batch["filename"][offset],
                        "label": batch["class_id"][offset],
                        "shard_name": shard_name,
                        "offset": offset,
                    }
                )

            print(
                f"[{dataset.split}] {shard_id + 1:03d}/{len(loader):03d} "
                f"{status}: {len(indices)} images; persisted to Drive"
            )

        index = (
            pd.DataFrame(records)
            .sort_values("dataset_index")
            .reset_index(drop=True)
        )
        if len(index) != len(dataset):
            raise RuntimeError("Cache index is incomplete.")

        _atomic_csv_save(index, index_path)
        summary_path = split_dir / "summary.json"
        summary_path.write_text(
            json.dumps(
                {
                    "split": dataset.split,
                    "num_images": len(dataset),
                    "num_shards": len(loader),
                },
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        self._persist_file(index_path)
        self._persist_file(summary_path)
        self._persist_file(self.local_cache_dir / "metadata.json")

        if not split_cache_complete(
            self.persistent_cache_dir,
            dataset.split,
            len(dataset),
        ):
            raise RuntimeError(
                f"Persistent Drive cache for {dataset.split!r} is incomplete."
            )

        print(
            f"{dataset.split}: cached {len(dataset):,} images locally and in Drive."
        )
        return index


## Restore persistent embeddings locally, then build only what is missing

The Drive cache is copied into `/content` first. The missing-cache check is then performed on the local copy. If all splits are complete, DINOv2 is never instantiated.


In [145]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
EXPECTED_IMAGES = {"train": 38_400, "val": 9_600, "test": 12_000}

# Restore every persistent shard to the fast local runtime.
copied_cache_files = copy_tree_update(DRIVE_CACHE_DIR, LOCAL_CACHE_DIR)
print(f"Feature-cache files copied from Drive to local: {copied_cache_files}")

missing_splits = [
    split
    for split, expected_images in EXPECTED_IMAGES.items()
    if not split_cache_complete(LOCAL_CACHE_DIR, split, expected_images)
]

if missing_splits:
    print("Missing/incomplete local cache splits:", missing_splits)

    dinov2_transform = build_dinov2_transform(224)
    raw_datasets = {
        split: MiniImageNetImageDataset(
            data_dir=LOCAL_DATA_DIR,
            image_dir=LOCAL_IMAGE_DIR,
            split=split,
            transform=dinov2_transform,
        )
        for split in missing_splits
    }

    # DINOv2 is loaded only because at least one persistent/local shard is missing.
    extractor = DINOv2FeatureExtractor(
        model_name="dinov2_vits14",
        image_size=224,
        output_dtype=torch.float16,
    )

    builder = DINOv2FeatureCacheBuilder(
        extractor=extractor,
        local_cache_dir=LOCAL_CACHE_DIR,
        persistent_cache_dir=DRIVE_CACHE_DIR,
        images_per_shard=256,
        extraction_batch_size=32,
        num_workers=2,
    )

    for split in missing_splits:
        builder.build(raw_datasets[split])

    del builder, extractor, raw_datasets
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("All feature caches were restored from Drive. DINOv2 was not loaded.")

# Final consistency check: both local and persistent copies must be complete.
for split, expected_images in EXPECTED_IMAGES.items():
    if not split_cache_complete(LOCAL_CACHE_DIR, split, expected_images):
        raise RuntimeError(f"Incomplete local cache for {split!r}.")
    if not split_cache_complete(DRIVE_CACHE_DIR, split, expected_images):
        raise RuntimeError(f"Incomplete persistent Drive cache for {split!r}.")

print("Local and persistent feature caches are complete.")


Mounted at /content/drive
Feature-cache files copied from Drive to local: 0
All feature caches were restored from Drive. DINOv2 was not loaded.
Local and persistent feature caches are complete.


## Cached feature dataset

Training reads feature shards from `LOCAL_CACHE_DIR`, not from the Drive mount. The Drive copy is used only for persistence across environments.


In [146]:
class MiniImageNetFeatureDataset(Dataset):
    def __init__(
        self,
        cache_dir: str | Path,
        split: str,
        max_cached_shards: int = 4,
    ) -> None:
        self.cache_dir = Path(cache_dir)
        self.split = split
        self.split_dir = self.cache_dir / split
        self.max_cached_shards = max_cached_shards

        metadata_path = self.cache_dir / "metadata.json"
        index_path = self.split_dir / "index.csv"
        if not metadata_path.exists() or not index_path.exists():
            raise FileNotFoundError(f"Incomplete cache for split {split!r}.")

        self.metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        self.index = pd.read_csv(index_path).sort_values("dataset_index").reset_index(drop=True)

        expected = EXPECTED_IMAGES[split]
        if len(self.index) != expected:
            raise ValueError(
                f"{split}: expected {expected:,} cached images, found {len(self.index):,}."
            )

        self.class_ids = sorted(self.index["label"].astype(str).unique().tolist())
        self.class_to_indices = {
            class_id: self.index.index[
                self.index["label"].astype(str) == class_id
            ].tolist()
            for class_id in self.class_ids
        }
        self._shards: OrderedDict[str, dict[str, torch.Tensor]] = OrderedDict()

    def __len__(self) -> int:
        return len(self.index)

    def indices_for_class(self, class_id: str) -> list[int]:
        return self.class_to_indices[class_id]

    def _load_shard(self, shard_name: str) -> dict[str, torch.Tensor]:
        if shard_name in self._shards:
            self._shards.move_to_end(shard_name)
            return self._shards[shard_name]

        shard = torch.load(
            self.split_dir / shard_name,
            map_location="cpu",
            weights_only=True,
        )
        self._shards[shard_name] = shard
        self._shards.move_to_end(shard_name)

        while len(self._shards) > self.max_cached_shards:
            self._shards.popitem(last=False)

        return shard

    def __getitem__(self, index: int) -> dict:
        row = self.index.iloc[index]
        shard = self._load_shard(row["shard_name"])
        offset = int(row["offset"])

        cached_index = int(shard["dataset_indices"][offset])
        if cached_index != index:
            raise RuntimeError(
                f"Index mismatch: requested {index}, shard contains {cached_index}."
            )

        return {
            "cls": shard["cls"][offset],
            "patches": shard["patches"][offset],
            "class_id": str(row["label"]),
            "filename": str(row["filename"]),
            "dataset_index": index,
        }


## Few-shot episodes from cached features

Support and query indices are sampled without replacement. The output contains DINOv2 features directly; no image decoding and no DINOv2 inference occur.


In [147]:
class FewShotFeatureEpisodeDataset(Dataset):
    def __init__(
        self,
        base_dataset: MiniImageNetFeatureDataset,
        n_way: int,
        k_shot: int,
        queries_per_class: int,
        num_episodes: int,
        seed: int,
        vary_by_epoch: bool,
    ) -> None:
        self.base_dataset = base_dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.queries_per_class = queries_per_class
        self.num_episodes = num_episodes
        self.seed = seed
        self.vary_by_epoch = vary_by_epoch
        self.epoch = 0

        if n_way > len(base_dataset.class_ids):
            raise ValueError("n_way exceeds the number of split classes.")
        if k_shot + queries_per_class > 600:
            raise ValueError("Not enough images per class.")

    def __len__(self) -> int:
        return self.num_episodes

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def _rng(self, episode_index: int) -> np.random.Generator:
        epoch = self.epoch if self.vary_by_epoch else 0
        return np.random.default_rng(
            np.random.SeedSequence([self.seed, epoch, episode_index])
        )

    def __getitem__(self, episode_index: int) -> dict:
        rng = self._rng(episode_index)
        class_ids = rng.choice(
            self.base_dataset.class_ids,
            size=self.n_way,
            replace=False,
        ).tolist()

        support_items, query_items = [], []
        support_indices, query_indices = [], []

        for class_id in class_ids:
            chosen = rng.choice(
                self.base_dataset.indices_for_class(class_id),
                size=self.k_shot + self.queries_per_class,
                replace=False,
            ).tolist()

            class_support_indices = chosen[: self.k_shot]
            class_query_indices = chosen[self.k_shot :]

            support_indices.append(class_support_indices)
            query_indices.append(class_query_indices)
            support_items.append(
                [self.base_dataset[index] for index in class_support_indices]
            )
            query_items.append(
                [self.base_dataset[index] for index in class_query_indices]
            )

        def stack(items: list[list[dict]], key: str) -> torch.Tensor:
            return torch.stack(
                [torch.stack([item[key] for item in class_items]) for class_items in items]
            )

        support_labels = torch.arange(self.n_way)[:, None].expand(
            self.n_way, self.k_shot
        ).clone()
        query_labels = torch.arange(self.n_way)[:, None].expand(
            self.n_way, self.queries_per_class
        ).clone()

        return {
            "support_cls": stack(support_items, "cls"),             # [N, K, D]
            "support_patches": stack(support_items, "patches"),     # [N, K, P, D]
            "query_cls": stack(query_items, "cls"),                 # [N, Q, D]
            "query_patches": stack(query_items, "patches"),         # [N, Q, P, D]
            "support_labels": support_labels,
            "query_labels": query_labels,
            "class_ids": class_ids,
            "support_indices": support_indices,
            "query_indices": query_indices,
            "episode_index": episode_index,
        }


def collate_feature_episodes(episodes: list[dict]) -> dict:
    tensor_keys = (
        "support_cls",
        "support_patches",
        "query_cls",
        "query_patches",
        "support_labels",
        "query_labels",
    )
    batch = {
        key: torch.stack([episode[key] for episode in episodes])
        for key in tensor_keys
    }
    for key in ("class_ids", "support_indices", "query_indices"):
        batch[key] = [episode[key] for episode in episodes]
    batch["episode_index"] = torch.tensor(
        [episode["episode_index"] for episode in episodes],
        dtype=torch.long,
    )
    return batch


## Load local embeddings and create episodes

At this point all embeddings have already been restored from Drive into the local runtime. No image decoding or DINOv2 inference occurs during episodic training.


In [148]:
train_features = MiniImageNetFeatureDataset(
    LOCAL_CACHE_DIR,
    "train",
    max_cached_shards=6,
)
val_features = MiniImageNetFeatureDataset(
    LOCAL_CACHE_DIR,
    "val",
    max_cached_shards=6,
)
test_features = MiniImageNetFeatureDataset(
    LOCAL_CACHE_DIR,
    "test",
    max_cached_shards=6,
)

train_episodes = FewShotFeatureEpisodeDataset(
    train_features,
    n_way=5,
    k_shot=5,
    queries_per_class=1,
    num_episodes=1_000,
    seed=42,
    vary_by_epoch=True,
)

val_episodes = FewShotFeatureEpisodeDataset(
    val_features,
    n_way=5,
    k_shot=5,
    queries_per_class=15,
    num_episodes=600,
    seed=10_000,
    vary_by_epoch=False,
)

test_episodes = FewShotFeatureEpisodeDataset(
    test_features,
    n_way=5,
    k_shot=5,
    queries_per_class=15,
    num_episodes=600,
    seed=20_000,
    vary_by_epoch=False,
)


In [149]:
episode = train_episodes[0]

print("Support CLS:", episode["support_cls"].shape)
print("Support patches:", episode["support_patches"].shape)
print("Query CLS:", episode["query_cls"].shape)
print("Query patches:", episode["query_patches"].shape)
print("Feature dtype:", episode["support_patches"].dtype)

support_index_set = {
    index for class_indices in episode["support_indices"] for index in class_indices
}
query_index_set = {
    index for class_indices in episode["query_indices"] for index in class_indices
}
assert support_index_set.isdisjoint(query_index_set)

print("Support/query indices are disjoint.")
print("No DINOv2 forward pass occurred in this check.")


Support CLS: torch.Size([5, 5, 384])
Support patches: torch.Size([5, 5, 256, 384])
Query CLS: torch.Size([5, 1, 384])
Query patches: torch.Size([5, 1, 256, 384])
Feature dtype: torch.float16
Support/query indices are disjoint.
No DINOv2 forward pass occurred in this check.


In [150]:
train_loader = DataLoader(
    train_episodes,
    batch_size=1,  # One episode per optimizer step initially.
    shuffle=False,
    num_workers=0,  # Start simple; increase only after profiling Drive I/O.
    collate_fn=collate_feature_episodes,
)

batch = next(iter(train_loader))
print("Meta-batched support patches:", batch["support_patches"].shape)
print("Meta-batched query patches:", batch["query_patches"].shape)


Meta-batched support patches: torch.Size([1, 5, 5, 256, 384])
Meta-batched query patches: torch.Size([1, 5, 1, 256, 384])


In [151]:
import importlib.util
import subprocess
import sys


if importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch_geometric",
        ]
    )

from torch_geometric.data import Batch, Data

In [152]:
from dataclasses import dataclass

import torch


@dataclass
class EpisodePatchGraphs:
    """
    Graphs created for one few-shot episode.

    Graph order is query-major, candidate-minor:

        query 0 with candidate 0
        query 0 with candidate 1
        ...
        query 1 with candidate 0
        ...

    Therefore, one scalar score per graph can later be reshaped as:

        [num_queries, num_candidates]
    """

    graphs: list[Data]

    # Correct episode class for each query.
    # Shape: [num_queries]
    targets: torch.Tensor

    num_queries: int
    num_candidates: int

In [153]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import torch
import torch.nn.functional as F
from torch_geometric.data import Data


class ClassConditionedPatchGraphBuilder:
    """
    Construct one graph from:

        one query image
        one candidate class containing K support images

    Node order
    ----------
    Query patches:
        nodes [0, P)

    Support image 0:
        nodes [P, 2P)

    Support image 1:
        nodes [2P, 3P)

    ...

    Support image K-1:
        nodes [KP, (K+1)P)

    Initial node feature
    --------------------
    x_i = [DINO_patch_i, normalized_row_i,
          normalized_column_i, is_query_i]

    With DINOv2 ViT-S/14:
        D = 384
        P = 256
        x dimension = 384 + 2 + 1 = 387

    Edge attributes
    ---------------
    edge_attr[:, 0] = cosine similarity
    edge_attr[:, 1] = is spatial edge
    edge_attr[:, 2] = is semantic cross-image edge
    edge_attr[:, 3] = relative grid-row displacement
    edge_attr[:, 4] = relative grid-column displacement
    """

    SPATIAL_EDGE = 0
    SEMANTIC_EDGE = 1

    def __init__(
        self,
        grid_size: tuple[int, int] = (16, 16),
        top_k: int = 10,
        min_similarity: float | None = None,
        graph_dtype: torch.dtype = torch.float32,
        similarity_device: str | torch.device | None = None,
    ) -> None:
        grid_height, grid_width = grid_size

        if grid_height <= 0 or grid_width <= 0:
            raise ValueError(
                f"Invalid grid size: {grid_size}."
            )

        if top_k <= 0:
            raise ValueError("top_k must be positive.")

        if (
            min_similarity is not None
            and not -1.0 <= min_similarity <= 1.0
        ):
            raise ValueError(
                "min_similarity must be between -1 and 1."
            )

        self.grid_size = grid_size
        self.num_patches = grid_height * grid_width
        self.top_k = top_k
        self.min_similarity = min_similarity
        self.graph_dtype = graph_dtype

        if similarity_device is None:
            similarity_device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.similarity_device = torch.device(
            similarity_device
        )

        self.coordinates = self._build_coordinates()

        (
            self.single_image_spatial_edges,
            self.single_image_displacements,
        ) = self._build_spatial_template()

    def _build_coordinates(self) -> torch.Tensor:
        """
        Create normalized grid coordinates.

        Returns
        -------
        Tensor[P, 2]

        Coordinates are ordered in the same row-major order
        as the DINOv2 patch tokens.
        """
        grid_height, grid_width = self.grid_size

        row_denominator = max(grid_height - 1, 1)
        column_denominator = max(grid_width - 1, 1)

        rows = (
            torch.arange(
                grid_height,
                dtype=torch.float32,
            )
            / row_denominator
        )

        columns = (
            torch.arange(
                grid_width,
                dtype=torch.float32,
            )
            / column_denominator
        )

        row_grid, column_grid = torch.meshgrid(
            rows,
            columns,
            indexing="ij",
        )

        coordinates = torch.stack(
            [
                row_grid.reshape(-1),
                column_grid.reshape(-1),
            ],
            dim=-1,
        )

        return coordinates

    def _build_spatial_template(
        self,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Construct bidirectional four-neighbour edges for one image.

        Every undirected relationship is represented by two entries:

            u -> v
            v -> u
        """
        grid_height, grid_width = self.grid_size

        sources: list[int] = []
        targets: list[int] = []
        displacements: list[list[float]] = []

        for row in range(grid_height):
            for column in range(grid_width):
                current = row * grid_width + column

                # Horizontal relationship:
                # current <-> patch to the right
                if column + 1 < grid_width:
                    right = current + 1

                    sources.extend([current, right])
                    targets.extend([right, current])

                    displacements.extend(
                        [
                            [0.0, 1.0],
                            [0.0, -1.0],
                        ]
                    )

                # Vertical relationship:
                # current <-> patch below
                if row + 1 < grid_height:
                    below = (
                        (row + 1) * grid_width
                        + column
                    )

                    sources.extend([current, below])
                    targets.extend([below, current])

                    displacements.extend(
                        [
                            [1.0, 0.0],
                            [-1.0, 0.0],
                        ]
                    )

        edge_index = torch.tensor(
            [sources, targets],
            dtype=torch.long,
        )

        displacement = torch.tensor(
            displacements,
            dtype=torch.float32,
        )

        return edge_index, displacement

    def _validate_inputs(
        self,
        query_patches: torch.Tensor,
        support_patches: torch.Tensor,
    ) -> None:
        if query_patches.ndim != 2:
            raise ValueError(
                "query_patches must have shape [P, D], "
                f"received {tuple(query_patches.shape)}."
            )

        if support_patches.ndim != 3:
            raise ValueError(
                "support_patches must have shape [K, P, D], "
                f"received {tuple(support_patches.shape)}."
            )

        if query_patches.shape[0] != self.num_patches:
            raise ValueError(
                f"Expected {self.num_patches} query patches, "
                f"received {query_patches.shape[0]}."
            )

        if support_patches.shape[1] != self.num_patches:
            raise ValueError(
                f"Expected {self.num_patches} support patches "
                f"per image, received "
                f"{support_patches.shape[1]}."
            )

        if (
            query_patches.shape[-1]
            != support_patches.shape[-1]
        ):
            raise ValueError(
                "Query and support embedding dimensions differ: "
                f"{query_patches.shape[-1]} versus "
                f"{support_patches.shape[-1]}."
            )

        if support_patches.shape[0] <= 0:
            raise ValueError(
                "At least one support image is required."
            )

        if not torch.isfinite(query_patches).all():
            raise ValueError(
                "Query patches contain NaN or infinity."
            )

        if not torch.isfinite(support_patches).all():
            raise ValueError(
                "Support patches contain NaN or infinity."
            )

    def _build_spatial_edges(
        self,
        patch_features: torch.Tensor,
        num_images: int,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
    ]:
        """
        Add four-neighbour edges independently to every image.
        """
        normalized_features = F.normalize(
            patch_features,
            p=2,
            dim=-1,
        )

        edge_indices = []
        edge_attributes = []
        edge_types = []

        base_edges = self.single_image_spatial_edges
        displacement = self.single_image_displacements

        for image_id in range(num_images):
            offset = image_id * self.num_patches

            image_edges = base_edges + offset
            source, target = image_edges

            cosine_similarity = (
                normalized_features[source]
                * normalized_features[target]
            ).sum(dim=-1)

            spatial_indicator = torch.ones_like(
                cosine_similarity
            )

            semantic_indicator = torch.zeros_like(
                cosine_similarity
            )

            image_edge_attributes = torch.stack(
                [
                    cosine_similarity,
                    spatial_indicator,
                    semantic_indicator,
                    displacement[:, 0],
                    displacement[:, 1],
                ],
                dim=-1,
            )

            edge_indices.append(image_edges)
            edge_attributes.append(
                image_edge_attributes
            )

            edge_types.append(
                torch.full(
                    size=(image_edges.shape[1],),
                    fill_value=self.SPATIAL_EDGE,
                    dtype=torch.long,
                )
            )

        return (
            torch.cat(edge_indices, dim=1),
            torch.cat(edge_attributes, dim=0),
            torch.cat(edge_types, dim=0),
        )

    def _build_semantic_edges(
        self,
        query_patches: torch.Tensor,
        support_patches: torch.Tensor,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
    ]:
        """
        Connect each query patch to its top-k support patches.

        The search is performed across all K support images of
        the candidate class:

            support: [K, P, D] -> [K*P, D]

        Similarity matrix:

            [P, D] @ [D, K*P] -> [P, K*P]
        """
        query = query_patches.to(
            device=self.similarity_device,
            dtype=torch.float32,
        )

        support = support_patches.reshape(
            -1,
            support_patches.shape[-1],
        ).to(
            device=self.similarity_device,
            dtype=torch.float32,
        )

        normalized_query = F.normalize(
            query,
            p=2,
            dim=-1,
        )

        normalized_support = F.normalize(
            support,
            p=2,
            dim=-1,
        )

        similarity_matrix = (
            normalized_query
            @ normalized_support.T
        )

        effective_top_k = min(
            self.top_k,
            normalized_support.shape[0],
        )

        selected_similarities, selected_support = (
            torch.topk(
                similarity_matrix,
                k=effective_top_k,
                dim=-1,
                largest=True,
                sorted=True,
            )
        )

        query_nodes = torch.arange(
            self.num_patches,
            device=self.similarity_device,
        ).unsqueeze(1).expand_as(selected_support)

        if self.min_similarity is not None:
            keep = (
                selected_similarities
                >= self.min_similarity
            )

            query_nodes = query_nodes[keep]
            selected_support = selected_support[keep]
            selected_similarities = (
                selected_similarities[keep]
            )

        else:
            query_nodes = query_nodes.reshape(-1)
            selected_support = (
                selected_support.reshape(-1)
            )
            selected_similarities = (
                selected_similarities.reshape(-1)
            )

        if selected_similarities.numel() == 0:
            raise RuntimeError(
                "No semantic edges survived graph construction. "
                "Reduce min_similarity or disable the threshold."
            )

        # Query occupies nodes [0, P).
        # Flattened support occupies nodes [P, P + K*P).
        global_support_nodes = (
            selected_support + self.num_patches
        )

        query_to_support = torch.stack(
            [
                query_nodes,
                global_support_nodes,
            ],
            dim=0,
        )

        support_to_query = torch.stack(
            [
                global_support_nodes,
                query_nodes,
            ],
            dim=0,
        )

        semantic_edge_index = torch.cat(
            [
                query_to_support,
                support_to_query,
            ],
            dim=1,
        ).cpu()

        # Both directions use the same visual similarity.
        bidirectional_similarities = torch.cat(
            [
                selected_similarities,
                selected_similarities,
            ],
            dim=0,
        ).cpu()

        zeros = torch.zeros_like(
            bidirectional_similarities
        )

        semantic_edge_attributes = torch.stack(
            [
                bidirectional_similarities,
                zeros,
                torch.ones_like(
                    bidirectional_similarities
                ),
                zeros,
                zeros,
            ],
            dim=-1,
        )

        semantic_edge_types = torch.full(
            size=(semantic_edge_index.shape[1],),
            fill_value=self.SEMANTIC_EDGE,
            dtype=torch.long,
        )

        return (
            semantic_edge_index,
            semantic_edge_attributes,
            semantic_edge_types,
            selected_similarities.detach().cpu(),
        )

    def build_graph(
        self,
        query_patches: torch.Tensor,
        support_patches: torch.Tensor,
        candidate_id: int | None = None,
        query_data_id: int | None = None,
        support_data_ids: Sequence[int] | None = None,
    ) -> Data:
        """
        Construct one query-candidate-class graph.

        Parameters
        ----------
        query_patches:
            [P, D]

        support_patches:
            [K, P, D]

        candidate_id:
            Candidate position in the current episode.
            This is metadata only and is never concatenated
            into node features.

        query_data_id, support_data_ids:
            Dataset-row identifiers used only for debugging and
            leakage verification.
        """
        self._validate_inputs(
            query_patches,
            support_patches,
        )

        # Cached embeddings are float16. Graph/GNN calculations
        # initially use float32.
        query_patches = (
            query_patches
            .detach()
            .to(
                device="cpu",
                dtype=torch.float32,
            )
            .contiguous()
        )

        support_patches = (
            support_patches
            .detach()
            .to(
                device="cpu",
                dtype=torch.float32,
            )
            .contiguous()
        )

        num_support_images = support_patches.shape[0]
        embedding_dim = query_patches.shape[-1]
        num_images = num_support_images + 1

        flattened_support = support_patches.reshape(
            num_support_images * self.num_patches,
            embedding_dim,
        )

        # [P + K*P, D]
        patch_features = torch.cat(
            [
                query_patches,
                flattened_support,
            ],
            dim=0,
        )

        num_nodes = patch_features.shape[0]

        # Repeat the same 16x16 coordinate system for each image.
        positions = self.coordinates.repeat(
            num_images,
            1,
        )

        node_is_query = torch.zeros(
            num_nodes,
            dtype=torch.bool,
        )

        node_is_query[: self.num_patches] = True

        query_indicator = (
            node_is_query
            .to(torch.float32)
            .unsqueeze(-1)
        )

        # 0 = query
        # 1...K = support images
        image_id = torch.arange(
            num_images,
            dtype=torch.long,
        ).repeat_interleave(self.num_patches)

        patch_id = torch.arange(
            self.num_patches,
            dtype=torch.long,
        ).repeat(num_images)

        # [num_nodes, D + 2 + 1]
        x = torch.cat(
            [
                patch_features,
                positions,
                query_indicator,
            ],
            dim=-1,
        ).to(self.graph_dtype)

        (
            spatial_edge_index,
            spatial_edge_attr,
            spatial_edge_type,
        ) = self._build_spatial_edges(
            patch_features=patch_features,
            num_images=num_images,
        )

        (
            semantic_edge_index,
            semantic_edge_attr,
            semantic_edge_type,
            selected_similarities,
        ) = self._build_semantic_edges(
            query_patches=query_patches,
            support_patches=support_patches,
        )

        edge_index = torch.cat(
            [
                spatial_edge_index,
                semantic_edge_index,
            ],
            dim=1,
        )

        edge_attr = torch.cat(
            [
                spatial_edge_attr,
                semantic_edge_attr,
            ],
            dim=0,
        ).to(self.graph_dtype)

        edge_type = torch.cat(
            [
                spatial_edge_type,
                semantic_edge_type,
            ],
            dim=0,
        )

        graph_arguments = {
            "x": x,
            "edge_index": edge_index,
            "edge_attr": edge_attr,
            "edge_type": edge_type,
            "pos": positions.to(self.graph_dtype),
            "node_is_query": node_is_query,
            "image_id": image_id,
            "patch_id": patch_id,

            # Graph-level metadata:
            "num_support_images": torch.tensor(
                [num_support_images],
                dtype=torch.long,
            ),
            "semantic_similarity_mean": torch.tensor(
                [selected_similarities.mean().item()],
                dtype=torch.float32,
            ),
            "semantic_similarity_min": torch.tensor(
                [selected_similarities.min().item()],
                dtype=torch.float32,
            ),
            "semantic_similarity_max": torch.tensor(
                [selected_similarities.max().item()],
                dtype=torch.float32,
            ),
        }

        if candidate_id is not None:
            graph_arguments["candidate_id"] = torch.tensor(
                [candidate_id],
                dtype=torch.long,
            )

        if query_data_id is not None:
            graph_arguments["query_data_id"] = torch.tensor(
                [query_data_id],
                dtype=torch.long,
            )

        if support_data_ids is not None:
            if (
                len(support_data_ids)
                != num_support_images
            ):
                raise ValueError(
                    "support_data_ids length must equal "
                    "the number of support images."
                )

            graph_arguments["support_data_ids"] = (
                torch.tensor(
                    list(support_data_ids),
                    dtype=torch.long,
                )
            )

        return Data(**graph_arguments)

    def build_episode_graphs(
        self,
        episode: dict,
    ) -> EpisodePatchGraphs:
        """
        Construct all query-candidate graphs in one episode.

        Input shapes
        ------------
        support_patches:
            [N, K, P, D]

        query_patches:
            [N, Q, P, D]

        Output graph count
        ------------------
            N * Q * N

        For 5-way and one query per class:
            5 * 1 * 5 = 25 graphs
        """
        support_patches = episode["support_patches"]
        query_patches = episode["query_patches"]
        query_labels = episode["query_labels"]

        if support_patches.ndim != 4:
            raise ValueError(
                "support_patches must have shape [N,K,P,D]."
            )

        if query_patches.ndim != 4:
            raise ValueError(
                "query_patches must have shape [N,Q,P,D]."
            )

        (
            n_way,
            k_shot,
            support_patch_count,
            embedding_dim,
        ) = support_patches.shape

        (
            query_class_count,
            queries_per_class,
            query_patch_count,
            query_embedding_dim,
        ) = query_patches.shape

        if query_class_count != n_way:
            raise ValueError(
                "Support and query class counts differ."
            )

        if support_patch_count != query_patch_count:
            raise ValueError(
                "Support and query patch counts differ."
            )

        if embedding_dim != query_embedding_dim:
            raise ValueError(
                "Support and query embedding dimensions differ."
            )

        graphs: list[Data] = []

        # The labels are read only for the eventual loss.
        # They are not passed into build_graph().
        targets = query_labels.reshape(-1).long().cpu()

        for query_class_position in range(n_way):
            for query_position in range(
                queries_per_class
            ):
                query_data_id = None

                if "query_indices" in episode:
                    query_data_id = int(
                        episode["query_indices"]
                        [query_class_position]
                        [query_position]
                    )

                for candidate_id in range(n_way):
                    support_data_ids = None

                    if "support_indices" in episode:
                        support_data_ids = (
                            episode["support_indices"]
                            [candidate_id]
                        )

                    graph = self.build_graph(
                        query_patches=(
                            query_patches[
                                query_class_position,
                                query_position,
                            ]
                        ),
                        support_patches=(
                            support_patches[candidate_id]
                        ),
                        candidate_id=candidate_id,
                        query_data_id=query_data_id,
                        support_data_ids=(
                            support_data_ids
                        ),
                    )

                    graphs.append(graph)

        num_queries = n_way * queries_per_class

        expected_graph_count = (
            num_queries * n_way
        )

        if len(graphs) != expected_graph_count:
            raise RuntimeError(
                f"Created {len(graphs)} graphs; "
                f"expected {expected_graph_count}."
            )

        return EpisodePatchGraphs(
            graphs=graphs,
            targets=targets,
            num_queries=num_queries,
            num_candidates=n_way,
        )

In [154]:
graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(
        train_features.metadata["grid_size"]
    ),
    top_k=10,
    min_similarity=None,
    graph_dtype=torch.float32,
    similarity_device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    ),
)

print("Grid size:", graph_builder.grid_size)
print("Patch count:", graph_builder.num_patches)
print(
    "Similarity device:",
    graph_builder.similarity_device,
)

Grid size: (16, 16)
Patch count: 256
Similarity device: cuda


In [155]:
episode = train_episodes[0]

query_class_position = 0
query_position = 0
candidate_id = 0

graph = graph_builder.build_graph(
    query_patches=episode["query_patches"][
        query_class_position,
        query_position,
    ],
    support_patches=episode["support_patches"][
        candidate_id
    ],
    candidate_id=candidate_id,
    query_data_id=episode["query_indices"][
        query_class_position
    ][query_position],
    support_data_ids=episode["support_indices"][
        candidate_id
    ],
)

print(graph)

Data(x=[1536, 387], edge_index=[2, 10880], edge_attr=[10880, 5], pos=[1536, 2], edge_type=[10880], node_is_query=[1536], image_id=[1536], patch_id=[1536], num_support_images=[1], semantic_similarity_mean=[1], semantic_similarity_min=[1], semantic_similarity_max=[1], candidate_id=[1], query_data_id=[1], support_data_ids=[5])


In [156]:
def validate_patch_graph(
    graph: Data,
    builder: ClassConditionedPatchGraphBuilder,
) -> None:
    num_support_images = int(
        graph.num_support_images.item()
    )

    num_images = num_support_images + 1
    num_patches = builder.num_patches

    expected_nodes = num_images * num_patches

    grid_height, grid_width = builder.grid_size

    # Directed four-neighbour entries for one image:
    spatial_edges_per_image = 2 * (
        grid_height * (grid_width - 1)
        + grid_width * (grid_height - 1)
    )

    expected_spatial_edges = (
        num_images * spatial_edges_per_image
    )

    expected_semantic_edges = 2 * (
        num_patches
        * min(
            builder.top_k,
            num_support_images * num_patches,
        )
    )

    assert graph.num_nodes == expected_nodes

    assert graph.x.shape == (
        expected_nodes,
        384 + 2 + 1,
    )

    assert graph.edge_index.shape[0] == 2
    assert graph.edge_attr.shape[1] == 5

    assert graph.edge_index.dtype == torch.long
    assert graph.edge_type.dtype == torch.long

    assert graph.edge_index.min() >= 0
    assert graph.edge_index.max() < graph.num_nodes

    assert torch.isfinite(graph.x).all()
    assert torch.isfinite(graph.edge_attr).all()

    # No self-loops were explicitly added.
    source, target = graph.edge_index

    assert not torch.any(source == target)

    spatial_mask = (
        graph.edge_type
        == builder.SPATIAL_EDGE
    )

    semantic_mask = (
        graph.edge_type
        == builder.SEMANTIC_EDGE
    )

    actual_spatial_edges = int(
        spatial_mask.sum()
    )

    actual_semantic_edges = int(
        semantic_mask.sum()
    )

    assert (
        actual_spatial_edges
        == expected_spatial_edges
    )

    if builder.min_similarity is None:
        assert (
            actual_semantic_edges
            == expected_semantic_edges
        )

    # Spatial edges must stay inside one image.
    spatial_source = source[spatial_mask]
    spatial_target = target[spatial_mask]

    assert torch.equal(
        graph.image_id[spatial_source],
        graph.image_id[spatial_target],
    )

    # Semantic edges must connect exactly one query node
    # and one support node.
    semantic_source = source[semantic_mask]
    semantic_target = target[semantic_mask]

    semantic_source_is_query = (
        graph.node_is_query[semantic_source]
    )

    semantic_target_is_query = (
        graph.node_is_query[semantic_target]
    )

    assert torch.all(
        semantic_source_is_query
        ^ semantic_target_is_query
    )

    # Each image contributes exactly P nodes.
    nodes_per_image = torch.bincount(
        graph.image_id,
        minlength=num_images,
    )

    assert torch.all(
        nodes_per_image == num_patches
    )

    # Query indicator is consistent in both metadata
    # and the final x feature.
    assert int(graph.node_is_query.sum()) == num_patches

    assert torch.equal(
        graph.x[:, -1].bool(),
        graph.node_is_query,
    )

    # Labels and original class names are not node features.
    assert getattr(graph, "y", None) is None
    assert getattr(graph, "class_ids", None) is None

    print("Graph validation passed.")
    print("Nodes:", graph.num_nodes)
    print("Node feature dimension:", graph.x.shape[1])
    print("Spatial edges:", actual_spatial_edges)
    print("Semantic edges:", actual_semantic_edges)
    print("Total edges:", graph.edge_index.shape[1])
    print(
        "Selected semantic similarity:",
        {
            "min": round(
                graph.semantic_similarity_min.item(),
                4,
            ),
            "mean": round(
                graph.semantic_similarity_mean.item(),
                4,
            ),
            "max": round(
                graph.semantic_similarity_max.item(),
                4,
            ),
        },
    )


validate_patch_graph(
    graph,
    graph_builder,
)

Graph validation passed.
Nodes: 1536
Node feature dimension: 387
Spatial edges: 5760
Semantic edges: 5120
Total edges: 10880
Selected semantic similarity: {'min': 0.2342, 'mean': 0.6499, 'max': 0.9074}


In [157]:
import time


start_time = time.perf_counter()

episode_graphs = (
    graph_builder.build_episode_graphs(
        episode
    )
)

elapsed = time.perf_counter() - start_time

print(
    "Number of query images:",
    episode_graphs.num_queries,
)

print(
    "Number of candidate classes:",
    episode_graphs.num_candidates,
)

print(
    "Number of graphs:",
    len(episode_graphs.graphs),
)

print(
    "Targets:",
    episode_graphs.targets,
)

print(
    f"Construction time: {elapsed:.3f} seconds"
)

Number of query images: 5
Number of candidate classes: 5
Number of graphs: 25
Targets: tensor([0, 1, 2, 3, 4])
Construction time: 0.561 seconds


In [158]:
first_query_graphs = episode_graphs.graphs[:5]

candidate_batch = Batch.from_data_list(
    first_query_graphs
)

print(
    "Number of batched graphs:",
    candidate_batch.num_graphs,
)

print(
    "Batched x:",
    candidate_batch.x.shape,
)

print(
    "Batched edge_index:",
    candidate_batch.edge_index.shape,
)

print(
    "Batched edge_attr:",
    candidate_batch.edge_attr.shape,
)

print(
    "Node-to-graph assignment:",
    candidate_batch.batch.shape,
)

print(
    "Candidate IDs:",
    candidate_batch.candidate_id,
)

Number of batched graphs: 5
Batched x: torch.Size([7680, 387])
Batched edge_index: torch.Size([2, 54400])
Batched edge_attr: torch.Size([54400, 5])
Node-to-graph assignment: torch.Size([7680])
Candidate IDs: tensor([0, 1, 2, 3, 4])


In [159]:
from __future__ import annotations

import torch
from torch import nn
from torch_geometric.nn import SAGEConv


class ResidualGraphSAGEBlock(nn.Module):
    """
    One residual GraphSAGE message-passing block.

    Input:
        h:          [num_nodes, hidden_dim]
        edge_index: [2, num_edges]

    Output:
        h:          [num_nodes, hidden_dim]
    """

    def __init__(
        self,
        hidden_dim: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if hidden_dim <= 0:
            raise ValueError("hidden_dim must be positive.")

        if not 0.0 <= dropout < 1.0:
            raise ValueError(
                "dropout must be in the range [0, 1)."
            )

        self.conv = SAGEConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            aggr="mean",
            normalize=False,
            root_weight=True,
        )

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(
        self,
        h: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Residual update:

            message = GraphSAGE(h, edge_index)
            output  = LayerNorm(h + Dropout(GELU(message)))
        """
        message = self.conv(
            h,
            edge_index,
        )

        message = self.activation(message)
        message = self.dropout(message)

        return self.norm(h + message)

In [160]:
class PatchGraphSAGEEncoder(nn.Module):
    """
    Refine patch-node representations with GraphSAGE.

    Default architecture:

        [num_nodes, 387]
            ↓ Linear
        [num_nodes, 256]
            ↓ LayerNorm + GELU + Dropout
        [num_nodes, 256]
            ↓ GraphSAGE block 1
        [num_nodes, 256]
            ↓ GraphSAGE block 2
        [num_nodes, 256]

    The current model uses edge_index but intentionally ignores
    edge_attr. Edge-aware message passing will be implemented as
    a separate model later.
    """

    def __init__(
        self,
        input_dim: int = 387,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if input_dim <= 0:
            raise ValueError("input_dim must be positive.")

        if hidden_dim <= 0:
            raise ValueError("hidden_dim must be positive.")

        if num_layers <= 0:
            raise ValueError("num_layers must be positive.")

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim,
        )

        self.input_norm = nn.LayerNorm(hidden_dim)
        self.input_activation = nn.GELU()
        self.input_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [
                ResidualGraphSAGEBlock(
                    hidden_dim=hidden_dim,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        graph,
        return_all_layers: bool = False,
    ):
        """
        Parameters
        ----------
        graph:
            PyG Data or Batch object containing:

                graph.x:          [num_nodes, input_dim]
                graph.edge_index: [2, num_edges]

        return_all_layers:
            When True, also return the representation after the
            input projection and after each GraphSAGE layer.

        Returns
        -------
        refined_nodes:
            [num_nodes, hidden_dim]

        layer_outputs, optional:
            List containing num_layers + 1 tensors.
        """
        x = graph.x
        edge_index = graph.edge_index

        if x.ndim != 2:
            raise ValueError(
                "graph.x must have shape [num_nodes, input_dim], "
                f"received {tuple(x.shape)}."
            )

        if x.shape[-1] != self.input_dim:
            raise ValueError(
                f"Expected node feature dimension {self.input_dim}, "
                f"received {x.shape[-1]}."
            )

        if edge_index.ndim != 2 or edge_index.shape[0] != 2:
            raise ValueError(
                "graph.edge_index must have shape [2, num_edges], "
                f"received {tuple(edge_index.shape)}."
            )

        if edge_index.dtype != torch.long:
            raise TypeError(
                "graph.edge_index must have dtype torch.long."
            )

        if not torch.isfinite(x).all():
            raise ValueError(
                "graph.x contains NaN or infinite values."
            )

        # Current graph construction already produces float32.
        # This also protects against accidentally passing cached float16
        # features directly into a float32 model.
        x = x.to(
            dtype=self.input_projection.weight.dtype
        )

        h = self.input_projection(x)
        h = self.input_norm(h)
        h = self.input_activation(h)
        h = self.input_dropout(h)

        layer_outputs = [h]

        for layer in self.layers:
            h = layer(
                h,
                edge_index,
            )

            layer_outputs.append(h)

        if return_all_layers:
            return h, layer_outputs

        return h

In [161]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = PatchGraphSAGEEncoder(
    input_dim=candidate_batch.x.shape[-1],  # 387
    hidden_dim=256,
    num_layers=2,
    dropout=0.1,
).to(device)

print(encoder)

PatchGraphSAGEEncoder(
  (input_projection): Linear(in_features=387, out_features=256, bias=True)
  (input_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (input_activation): GELU(approximate='none')
  (input_dropout): Dropout(p=0.1, inplace=False)
  (layers): ModuleList(
    (0-1): 2 x ResidualGraphSAGEBlock(
      (conv): SAGEConv(256, 256, aggr=mean)
      (activation): GELU(approximate='none')
      (dropout): Dropout(p=0.1, inplace=False)
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
)


In [162]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
    if parameter.requires_grad
)

print(
    "Trainable GraphSAGE parameters:",
    f"{trainable_parameters:,}",
)

Trainable GraphSAGE parameters: 363,520


In [163]:
candidate_batch_device = candidate_batch.clone().to(device)

encoder.eval()

with torch.no_grad():
    refined_nodes, layer_outputs = encoder(
        candidate_batch_device,
        return_all_layers=True,
    )

print(
    "Input nodes:",
    candidate_batch_device.x.shape,
)

for layer_index, layer_output in enumerate(
    layer_outputs
):
    print(
        f"Representation {layer_index}:",
        layer_output.shape,
    )

print(
    "Final refined nodes:",
    refined_nodes.shape,
)

Input nodes: torch.Size([7680, 387])
Representation 0: torch.Size([7680, 256])
Representation 1: torch.Size([7680, 256])
Representation 2: torch.Size([7680, 256])
Final refined nodes: torch.Size([7680, 256])


In [164]:
encoder.eval()

x_before = (
    candidate_batch_device.x
    .detach()
    .clone()
)

with torch.no_grad():
    _ = encoder(candidate_batch_device)

assert torch.equal(
    candidate_batch_device.x,
    x_before,
)

print("Original graph node features were not modified.")

Original graph node features were not modified.


In [165]:
encoder.train()
encoder.zero_grad(set_to_none=True)

refined_nodes = encoder(
    candidate_batch_device
)

# Temporary diagnostic loss.
# This is not the eventual classification loss.
diagnostic_loss = refined_nodes.square().mean()

diagnostic_loss.backward()

parameters_with_gradients = 0
total_gradient_norm = 0.0

for name, parameter in encoder.named_parameters():
    if not parameter.requires_grad:
        continue

    if parameter.grad is None:
        raise RuntimeError(
            f"No gradient was produced for parameter {name!r}."
        )

    if not torch.isfinite(parameter.grad).all():
        raise RuntimeError(
            f"Non-finite gradient detected in {name!r}."
        )

    parameters_with_gradients += 1
    total_gradient_norm += (
        parameter.grad.detach().norm().item()
    )

print(
    "Diagnostic loss:",
    diagnostic_loss.item(),
)

print(
    "Parameters receiving gradients:",
    parameters_with_gradients,
)

print(
    "Total gradient norm:",
    total_gradient_norm,
)

assert total_gradient_norm > 0.0

print("Gradient-flow test passed.")

Diagnostic loss: 0.9999915957450867
Parameters receiving gradients: 14
Total gradient norm: 0.17523648629861555
Gradient-flow test passed.


In [166]:
encoder.eval()

single_graph = first_query_graphs[0].clone().to(device)

with torch.no_grad():
    single_output = encoder(
        single_graph
    )

    batched_output = encoder(
        candidate_batch_device
    )

# PyG's batch vector identifies which graph each node belongs to.
graph_zero_mask = (
    candidate_batch_device.batch == 0
)

batched_graph_zero_output = batched_output[
    graph_zero_mask
]

print(
    "Single graph output:",
    single_output.shape,
)

print(
    "Graph 0 inside batch:",
    batched_graph_zero_output.shape,
)

torch.testing.assert_close(
    single_output,
    batched_graph_zero_output,
    rtol=1e-4,
    atol=1e-5,
)

print(
    "Graph-isolation test passed: "
    "batched graphs do not exchange messages."
)

Single graph output: torch.Size([1536, 256])
Graph 0 inside batch: torch.Size([1536, 256])
Graph-isolation test passed: batched graphs do not exchange messages.


In [167]:
encoder.eval()

with torch.no_grad():
    refined_nodes, layer_outputs = encoder(
        candidate_batch_device,
        return_all_layers=True,
    )

projected_nodes = layer_outputs[0]

mean_absolute_change = (
    refined_nodes - projected_nodes
).abs().mean()

mean_cosine_similarity = (
    torch.nn.functional.cosine_similarity(
        projected_nodes,
        refined_nodes,
        dim=-1,
    )
).mean()

print(
    "Mean absolute representation change:",
    mean_absolute_change.item(),
)

print(
    "Mean projected/refined cosine similarity:",
    mean_cosine_similarity.item(),
)

Mean absolute representation change: 0.5563485622406006
Mean projected/refined cosine similarity: 0.7624136805534363


In [168]:
from dataclasses import dataclass

import torch


@dataclass
class CandidateReadoutOutput:
    """
    Readout results for G query-candidate graphs.
    """

    # [G, hidden_dim]
    query_embeddings: torch.Tensor

    # [G, K, hidden_dim]
    support_image_embeddings: torch.Tensor

    # [G, hidden_dim]
    prototypes: torch.Tensor

    # Cosine values before temperature scaling: [G]
    cosine_similarities: torch.Tensor

    # Temperature-scaled scores: [G]
    scores: torch.Tensor

In [169]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import global_mean_pool


class MeanPrototypeCosineReadout(nn.Module):
    """
    Mean-pool refined patch nodes into image embeddings,
    average support-image embeddings into a class prototype,
    and calculate one cosine score per graph.

    Node metadata expected in the PyG Batch:
        graph_batch.batch:    [num_nodes]
        graph_batch.image_id: [num_nodes]

    Local image IDs:
        0       = query image
        1 ... K = support images
    """

    def __init__(
        self,
        temperature: float = 0.1,
        learnable_temperature: bool = False,
    ) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError(
                "temperature must be greater than zero."
            )

        initial_logit_scale = torch.log(
            torch.tensor(
                1.0 / temperature,
                dtype=torch.float32,
            )
        )

        if learnable_temperature:
            self.logit_scale = nn.Parameter(
                initial_logit_scale
            )
        else:
            self.register_buffer(
                "logit_scale",
                initial_logit_scale,
            )

        self.learnable_temperature = (
            learnable_temperature
        )

    @property
    def temperature(self) -> torch.Tensor:
        """
        Return T = 1 / scale.
        """
        return 1.0 / self.logit_scale.exp()

    def forward(
        self,
        refined_nodes: torch.Tensor,
        graph_batch,
    ) -> CandidateReadoutOutput:
        """
        Parameters
        ----------
        refined_nodes:
            Refined node representations:
            [total_nodes, hidden_dim]

        graph_batch:
            PyG Batch containing G graphs.

        Returns
        -------
        CandidateReadoutOutput
        """
        if refined_nodes.ndim != 2:
            raise ValueError(
                "refined_nodes must have shape "
                "[total_nodes, hidden_dim]."
            )

        if refined_nodes.shape[0] != graph_batch.num_nodes:
            raise ValueError(
                "Number of refined nodes does not match "
                "graph_batch.num_nodes."
            )

        if not hasattr(graph_batch, "batch"):
            raise ValueError(
                "graph_batch must contain a node-to-graph "
                "assignment vector named 'batch'."
            )

        if not hasattr(graph_batch, "image_id"):
            raise ValueError(
                "graph_batch must contain image_id metadata."
            )

        if not hasattr(
            graph_batch,
            "num_support_images",
        ):
            raise ValueError(
                "graph_batch must contain "
                "num_support_images metadata."
            )

        if not torch.isfinite(refined_nodes).all():
            raise ValueError(
                "refined_nodes contains NaN or infinity."
            )

        graph_ids = graph_batch.batch.long()
        local_image_ids = graph_batch.image_id.long()

        num_graphs = graph_batch.num_graphs

        support_counts = (
            graph_batch.num_support_images
            .reshape(-1)
            .long()
        )

        if support_counts.numel() != num_graphs:
            raise ValueError(
                "Expected one num_support_images value "
                "for every graph."
            )

        if not torch.all(
            support_counts == support_counts[0]
        ):
            raise ValueError(
                "All graphs in one batch must currently "
                "have the same number of support images."
            )

        num_support_images = int(
            support_counts[0].item()
        )

        num_images_per_graph = (
            num_support_images + 1
        )

        if local_image_ids.min() < 0:
            raise ValueError(
                "image_id values cannot be negative."
            )

        if (
            local_image_ids.max()
            >= num_images_per_graph
        ):
            raise ValueError(
                "image_id contains a value outside the "
                "expected query/support image range."
            )

        # Convert local image IDs into batch-global image IDs.
        #
        # Graph 0:
        #   query=0, supports=1...K
        #
        # Graph 1:
        #   query=K+1, supports=K+2...2(K+1)-1
        #
        # etc.
        global_image_ids = (
            graph_ids * num_images_per_graph
            + local_image_ids
        )

        total_images = (
            num_graphs * num_images_per_graph
        )

        # [G * (K+1), hidden_dim]
        pooled_images = global_mean_pool(
            x=refined_nodes,
            batch=global_image_ids,
            size=total_images,
        )

        # [G, K+1, hidden_dim]
        pooled_images = pooled_images.reshape(
            num_graphs,
            num_images_per_graph,
            refined_nodes.shape[-1],
        )

        # image_id 0 is always the query.
        query_embeddings = pooled_images[:, 0]

        # image_id 1...K are support images.
        support_image_embeddings = (
            pooled_images[:, 1:]
        )

        # [G, hidden_dim]
        prototypes = (
            support_image_embeddings.mean(dim=1)
        )

        normalized_queries = F.normalize(
            query_embeddings,
            p=2,
            dim=-1,
        )

        normalized_prototypes = F.normalize(
            prototypes,
            p=2,
            dim=-1,
        )

        cosine_similarities = (
            normalized_queries
            * normalized_prototypes
        ).sum(dim=-1)

        # Prevent uncontrolled scale growth if temperature
        # becomes learnable later.
        scale = self.logit_scale.exp().clamp(
            max=100.0
        )

        scores = cosine_similarities * scale

        return CandidateReadoutOutput(
            query_embeddings=query_embeddings,
            support_image_embeddings=(
                support_image_embeddings
            ),
            prototypes=prototypes,
            cosine_similarities=(
                cosine_similarities
            ),
            scores=scores,
        )

In [170]:
readout = MeanPrototypeCosineReadout(
    temperature=0.1,
    learnable_temperature=False,
).to(device)

In [171]:
encoder.eval()
readout.eval()

candidate_batch_device = candidate_batch.to(
    device
)

with torch.no_grad():
    refined_nodes = encoder(
        candidate_batch_device
    )

    readout_output = readout(
        refined_nodes=refined_nodes,
        graph_batch=candidate_batch_device,
    )

print(
    "Refined nodes:",
    refined_nodes.shape,
)

print(
    "Query embeddings:",
    readout_output.query_embeddings.shape,
)

print(
    "Support image embeddings:",
    readout_output
    .support_image_embeddings.shape,
)

print(
    "Prototypes:",
    readout_output.prototypes.shape,
)

print(
    "Cosine similarities:",
    readout_output.cosine_similarities.shape,
)

print(
    "Scores:",
    readout_output.scores.shape,
)

print(
    "Temperature:",
    readout.temperature.item(),
)

Refined nodes: torch.Size([7680, 256])
Query embeddings: torch.Size([5, 256])
Support image embeddings: torch.Size([5, 5, 256])
Prototypes: torch.Size([5, 256])
Cosine similarities: torch.Size([5])
Scores: torch.Size([5])
Temperature: 0.10000000149011612


In [172]:
probabilities = torch.softmax(
    readout_output.scores,
    dim=-1,
)

predicted_candidate = int(
    probabilities.argmax().item()
)

print(
    "Raw cosine similarities:",
    readout_output.cosine_similarities,
)

print(
    "Temperature-scaled scores:",
    readout_output.scores,
)

print(
    "Candidate probabilities:",
    probabilities,
)

print(
    "Predicted candidate:",
    predicted_candidate,
)

Raw cosine similarities: tensor([0.8044, 0.2659, 0.2314, 0.4902, 0.2552], device='cuda:0')
Temperature-scaled scores: tensor([8.0443, 2.6592, 2.3137, 4.9018, 2.5522], device='cuda:0')
Candidate probabilities: tensor([0.9478, 0.0043, 0.0031, 0.0409, 0.0039], device='cuda:0')
Predicted candidate: 0


In [173]:
def validate_readout(
    output: CandidateReadoutOutput,
    graph_batch,
    hidden_dim: int,
) -> None:
    num_graphs = graph_batch.num_graphs

    support_counts = (
        graph_batch.num_support_images
        .reshape(-1)
        .long()
    )

    num_support_images = int(
        support_counts[0].item()
    )

    assert output.query_embeddings.shape == (
        num_graphs,
        hidden_dim,
    )

    assert (
        output.support_image_embeddings.shape
        == (
            num_graphs,
            num_support_images,
            hidden_dim,
        )
    )

    assert output.prototypes.shape == (
        num_graphs,
        hidden_dim,
    )

    assert output.cosine_similarities.shape == (
        num_graphs,
    )

    assert output.scores.shape == (
        num_graphs,
    )

    assert torch.isfinite(
        output.query_embeddings
    ).all()

    assert torch.isfinite(
        output.support_image_embeddings
    ).all()

    assert torch.isfinite(
        output.prototypes
    ).all()

    assert torch.isfinite(
        output.scores
    ).all()

    assert torch.all(
        output.cosine_similarities >= -1.0001
    )

    assert torch.all(
        output.cosine_similarities <= 1.0001
    )

    print("Readout validation passed.")


validate_readout(
    output=readout_output,
    graph_batch=candidate_batch_device,
    hidden_dim=encoder.hidden_dim,
)

Readout validation passed.


In [174]:
manual_prototype = (
    readout_output
    .support_image_embeddings[0]
    .mean(dim=0)
)

torch.testing.assert_close(
    manual_prototype,
    readout_output.prototypes[0],
)

print(
    "Support prototype equals the mean of "
    "the five support-image embeddings."
)

Support prototype equals the mean of the five support-image embeddings.


In [175]:
query_embeddings = (
    readout_output.query_embeddings
)

query_similarity_matrix = (
    F.normalize(query_embeddings, dim=-1)
    @ F.normalize(query_embeddings, dim=-1).T
)

print(
    "Candidate-conditioned query "
    "similarity matrix:\n",
    query_similarity_matrix,
)

Candidate-conditioned query similarity matrix:
 tensor([[1.0000, 0.9338, 0.9304, 0.9271, 0.9348],
        [0.9338, 1.0000, 0.9765, 0.9612, 0.9841],
        [0.9304, 0.9765, 1.0000, 0.9512, 0.9675],
        [0.9271, 0.9612, 0.9512, 1.0000, 0.9518],
        [0.9348, 0.9841, 0.9675, 0.9518, 1.0000]], device='cuda:0')


In [176]:
class CrossImageGraphMatcher(nn.Module):
    """
    Complete trainable graph matching module:

        graph
        → GraphSAGE
        → image mean pooling
        → support prototype
        → cosine score
    """

    def __init__(
        self,
        input_dim: int = 387,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.1,
        temperature: float = 0.1,
        learnable_temperature: bool = False,
    ) -> None:
        super().__init__()

        self.encoder = PatchGraphSAGEEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
        )

        self.readout = MeanPrototypeCosineReadout(
            temperature=temperature,
            learnable_temperature=(
                learnable_temperature
            ),
        )

    def forward(
        self,
        graph_batch,
        return_embeddings: bool = False,
    ):
        refined_nodes = self.encoder(
            graph_batch
        )

        output = self.readout(
            refined_nodes=refined_nodes,
            graph_batch=graph_batch,
        )

        if return_embeddings:
            return output

        return output.scores

In [177]:
model = CrossImageGraphMatcher(
    input_dim=387,
    hidden_dim=256,
    num_layers=2,
    dropout=0.1,
    temperature=0.1,
    learnable_temperature=False,
).to(device)

In [178]:
model.eval()

with torch.no_grad():
    scores = model(
        candidate_batch_device
    )

print("Candidate scores:", scores)
print("Score shape:", scores.shape)

assert scores.shape == (5,)

Candidate scores: tensor([8.5422, 1.6233, 1.8770, 3.4539, 2.7347], device='cuda:0')
Score shape: torch.Size([5])


In [179]:
from dataclasses import dataclass

import torch


@dataclass
class EpisodePrediction:
    """
    Prediction results for one few-shot episode.
    """

    # [num_queries, num_candidates]
    logits: torch.Tensor

    # [num_queries]
    targets: torch.Tensor

    # [num_queries]
    predictions: torch.Tensor

    # Scalar
    loss: torch.Tensor

    # Scalar in [0, 1]
    accuracy: torch.Tensor

In [180]:
from torch_geometric.data import Batch


def score_graph_list(
    model: torch.nn.Module,
    graphs: list,
    device: torch.device,
    graph_microbatch_size: int = 2,
) -> torch.Tensor:
    if graph_microbatch_size <= 0:
        raise ValueError(
            "graph_microbatch_size must be positive."
        )

    if len(graphs) == 0:
        raise ValueError(
            "At least one graph is required."
        )

    score_chunks = []

    for start in range(
        0,
        len(graphs),
        graph_microbatch_size,
    ):
        end = min(
            start + graph_microbatch_size,
            len(graphs),
        )

        # Clone and normalize all graphs to CPU before collation.
        # Only the completed Batch is transferred to the GPU.
        graph_chunk = [
            graph.clone().cpu()
            for graph in graphs[start:end]
        ]

        graph_batch = Batch.from_data_list(
            graph_chunk
        ).to(device)

        chunk_scores = model(graph_batch)

        expected_shape = (end - start,)

        if tuple(chunk_scores.shape) != expected_shape:
            raise RuntimeError(
                f"Model returned scores with shape "
                f"{tuple(chunk_scores.shape)}; expected "
                f"{expected_shape}."
            )

        score_chunks.append(chunk_scores)

    return torch.cat(score_chunks, dim=0)

In [181]:
import torch.nn.functional as F


def predict_episode(
    model: torch.nn.Module,
    episode_graphs: EpisodePatchGraphs,
    device: torch.device,
    graph_microbatch_size: int = 2,
) -> EpisodePrediction:
    """
    Score all query-candidate graphs and calculate episodic
    cross-entropy.

    This builds one autograd graph for the complete episode.
    For lower-memory training, use train_one_episode() below.
    """
    expected_graph_count = (
        episode_graphs.num_queries
        * episode_graphs.num_candidates
    )

    if len(episode_graphs.graphs) != expected_graph_count:
        raise ValueError(
            f"Episode contains {len(episode_graphs.graphs)} "
            f"graphs; expected {expected_graph_count}."
        )

    flat_scores = score_graph_list(
        model=model,
        graphs=episode_graphs.graphs,
        device=device,
        graph_microbatch_size=graph_microbatch_size,
    )

    logits = flat_scores.reshape(
        episode_graphs.num_queries,
        episode_graphs.num_candidates,
    )

    targets = episode_graphs.targets.to(
        device=device,
        dtype=torch.long,
    )

    if targets.shape != (
        episode_graphs.num_queries,
    ):
        raise ValueError(
            f"Targets have shape {tuple(targets.shape)}; "
            f"expected "
            f"{(episode_graphs.num_queries,)}."
        )

    loss = F.cross_entropy(
        logits,
        targets,
    )

    predictions = logits.argmax(dim=-1)

    accuracy = (
        predictions == targets
    ).float().mean()

    return EpisodePrediction(
        logits=logits,
        targets=targets,
        predictions=predictions,
        loss=loss,
        accuracy=accuracy,
    )

In [182]:
model.eval()

with torch.no_grad():
    prediction = predict_episode(
        model=model,
        episode_graphs=episode_graphs,
        device=device,
        graph_microbatch_size=2,
    )

print("Logits shape:", prediction.logits.shape)
print("Targets:", prediction.targets)
print("Predictions:", prediction.predictions)
print("Loss:", prediction.loss.item())
print("Accuracy:", prediction.accuracy.item())

Logits shape: torch.Size([5, 5])
Targets: tensor([0, 1, 2, 3, 4], device='cuda:0')
Predictions: tensor([0, 1, 2, 3, 2], device='cuda:0')
Loss: 0.5895649790763855
Accuracy: 0.800000011920929


In [183]:
probabilities = torch.softmax(
    prediction.logits,
    dim=-1,
)

print("Logits:\n", prediction.logits)
print("Probabilities:\n", probabilities)

Logits:
 tensor([[8.5422e+00, 1.6233e+00, 1.8770e+00, 3.4539e+00, 2.7347e+00],
        [5.7531e-03, 6.4237e+00, 4.2280e+00, 2.3447e+00, 3.3133e+00],
        [2.0445e+00, 5.5898e+00, 7.6237e+00, 2.7554e+00, 4.1017e+00],
        [4.2691e+00, 2.6401e+00, 3.7189e+00, 4.2750e+00, 3.6881e+00],
        [2.2311e+00, 4.4296e+00, 4.6434e+00, 3.4152e+00, 4.2872e+00]],
       device='cuda:0')
Probabilities:
 tensor([[9.8869e-01, 9.7777e-04, 1.2601e-03, 6.0984e-03, 2.9711e-03],
        [1.3896e-03, 8.5149e-01, 9.4750e-02, 1.4410e-02, 3.7960e-02],
        [3.2221e-03, 1.1164e-01, 8.5337e-01, 6.5592e-03, 2.5209e-02],
        [2.9956e-01, 5.8751e-02, 1.7279e-01, 3.0133e-01, 1.6756e-01],
        [3.1004e-02, 2.7939e-01, 3.4600e-01, 1.0131e-01, 2.4230e-01]],
       device='cuda:0')


In [184]:
torch.testing.assert_close(
    probabilities.sum(dim=-1),
    torch.ones(
        episode_graphs.num_queries,
        device=device,
    ),
)

print("Probability validation passed.")

Probability validation passed.


In [185]:
@dataclass
class EpisodeTrainingResult:
    loss: float
    accuracy: float
    logits: torch.Tensor
    targets: torch.Tensor
    predictions: torch.Tensor
    gradient_norm: float

In [186]:
from torch.nn.utils import clip_grad_norm_


def train_one_episode(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    episode_graphs: EpisodePatchGraphs,
    device: torch.device,
    graph_microbatch_size: int = 2,
    max_gradient_norm: float | None = 1.0,
) -> EpisodeTrainingResult:
    """
    Perform one optimizer update using one few-shot episode.

    Only the five candidate graphs associated with one query
    need to retain activations simultaneously.
    """
    model.train()
    optimizer.zero_grad(set_to_none=True)

    num_queries = episode_graphs.num_queries
    num_candidates = episode_graphs.num_candidates

    expected_graph_count = (
        num_queries * num_candidates
    )

    if len(episode_graphs.graphs) != expected_graph_count:
        raise ValueError(
            f"Episode contains {len(episode_graphs.graphs)} "
            f"graphs; expected {expected_graph_count}."
        )

    targets = episode_graphs.targets.to(
        device=device,
        dtype=torch.long,
    )

    detached_logits = []
    detached_losses = []

    correct_predictions = 0

    for query_index in range(num_queries):
        start = query_index * num_candidates
        end = start + num_candidates

        query_candidate_graphs = (
            episode_graphs.graphs[start:end]
        )

        # [num_candidates]
        query_scores = score_graph_list(
            model=model,
            graphs=query_candidate_graphs,
            device=device,
            graph_microbatch_size=(
                graph_microbatch_size
            ),
        )

        # Cross-entropy expects [batch_size, classes].
        query_logits = query_scores.unsqueeze(0)

        query_target = targets[
            query_index : query_index + 1
        ]

        query_loss = F.cross_entropy(
            query_logits,
            query_target,
        )

        # Dividing by num_queries makes the accumulated gradient
        # equal to the gradient of the mean episode loss.
        normalized_query_loss = (
            query_loss / num_queries
        )

        normalized_query_loss.backward()

        query_prediction = query_scores.argmax()

        correct_predictions += int(
            query_prediction == query_target[0]
        )

        detached_logits.append(
            query_scores.detach().cpu()
        )

        detached_losses.append(
            query_loss.detach().cpu()
        )

    if max_gradient_norm is not None:
        if max_gradient_norm <= 0:
            raise ValueError(
                "max_gradient_norm must be positive "
                "or None."
            )

        gradient_norm = clip_grad_norm_(
            model.parameters(),
            max_norm=max_gradient_norm,
        )

        gradient_norm_value = float(
            gradient_norm.item()
        )

    else:
        squared_gradient_norm = 0.0

        for parameter in model.parameters():
            if parameter.grad is not None:
                squared_gradient_norm += (
                    parameter.grad.detach().norm().item()
                    ** 2
                )

        gradient_norm_value = (
            squared_gradient_norm ** 0.5
        )

    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            continue

        if not torch.isfinite(parameter.grad).all():
            raise RuntimeError(
                f"Non-finite gradient detected in {name!r}."
            )

    optimizer.step()

    logits = torch.stack(
        detached_logits,
        dim=0,
    )

    mean_loss = torch.stack(
        detached_losses
    ).mean().item()

    predictions = logits.argmax(dim=-1)

    accuracy = (
        correct_predictions / num_queries
    )

    return EpisodeTrainingResult(
        loss=mean_loss,
        accuracy=accuracy,
        logits=logits,
        targets=targets.detach().cpu(),
        predictions=predictions,
        gradient_norm=gradient_norm_value,
    )

In [187]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

In [188]:
parameter_before = (
    next(model.parameters())
    .detach()
    .clone()
)

In [189]:
training_result = train_one_episode(
    model=model,
    optimizer=optimizer,
    episode_graphs=episode_graphs,
    device=device,
    graph_microbatch_size=2,
    max_gradient_norm=1.0,
)

print("Episode loss:", training_result.loss)
print("Episode accuracy:", training_result.accuracy)
print("Predictions:", training_result.predictions)
print("Targets:", training_result.targets)
print(
    "Gradient norm before clipping:",
    training_result.gradient_norm,
)

Episode loss: 0.5878525972366333
Episode accuracy: 0.6
Predictions: tensor([0, 1, 2, 0, 2])
Targets: tensor([0, 1, 2, 3, 4])
Gradient norm before clipping: 6.971317291259766


In [190]:
parameter_after = (
    next(model.parameters())
    .detach()
    .clone()
)

assert not torch.equal(
    parameter_before,
    parameter_after,
)

print("Optimizer-step validation passed.")

Optimizer-step validation passed.


In [191]:
manual_loss = F.cross_entropy(
    training_result.logits,
    training_result.targets,
)

print("Returned loss:", training_result.loss)
print("Manual loss:", manual_loss.item())

assert abs(
    training_result.loss
    - manual_loss.item()
) < 1e-5

print("Episode-loss validation passed.")

Returned loss: 0.5878525972366333
Manual loss: 0.5878525972366333
Episode-loss validation passed.


In [205]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os

import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch_geometric.data import Data


def _get_2d_value(values, row: int, column: int):
    """
    Support both:
        Tensor[N, Q]
        nested list[N][Q]
    """
    if isinstance(values, torch.Tensor):
        return values[row, column]

    return values[row][column]


def build_query_candidate_graphs(
    graph_builder: ClassConditionedPatchGraphBuilder,
    episode: dict,
    query_class_position: int,
    query_position: int,
) -> QueryCandidateGraphs:
    support_patches = episode["support_patches"]
    query_patches = episode["query_patches"]
    query_labels = episode["query_labels"]

    if support_patches.ndim != 4:
        raise ValueError(
            "support_patches must have shape [N, K, P, D]."
        )

    if query_patches.ndim != 4:
        raise ValueError(
            "query_patches must have shape [N, Q, P, D]."
        )

    n_way = support_patches.shape[0]
    queries_per_class = query_patches.shape[1]

    if not 0 <= query_class_position < n_way:
        raise IndexError(
            "query_class_position is outside the episode."
        )

    if not 0 <= query_position < queries_per_class:
        raise IndexError(
            "query_position is outside the episode."
        )

    query_data_id = None

    if "query_indices" in episode:
        query_data_id = _to_python_int(
            _get_2d_value(
                episode["query_indices"],
                query_class_position,
                query_position,
            )
        )

    graphs = []

    for candidate_id in range(n_way):
        support_data_ids = None

        if "support_indices" in episode:
            support_data_ids = _to_python_int_list(
                episode["support_indices"][candidate_id]
            )

        graph = graph_builder.build_graph(
            query_patches=query_patches[
                query_class_position,
                query_position,
            ],
            support_patches=support_patches[
                candidate_id
            ],
            candidate_id=candidate_id,
            query_data_id=query_data_id,
            support_data_ids=support_data_ids,
        )

        graphs.append(graph)

    target = _to_python_int(
        _get_2d_value(
            query_labels,
            query_class_position,
            query_position,
        )
    )

    if not 0 <= target < n_way:
        raise ValueError(
            f"Invalid episode target {target} "
            f"for {n_way}-way episode."
        )

    return QueryCandidateGraphs(
        graphs=graphs,
        target=target,
        query_class_position=query_class_position,
        query_position=query_position,
    )

In [193]:
@dataclass
class EpisodeRunResult:
    loss: float
    accuracy: float
    correct: int
    num_queries: int
    logits: torch.Tensor
    targets: torch.Tensor
    gradient_norm: float | None = None

In [194]:
def train_feature_episode(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    graph_builder: ClassConditionedPatchGraphBuilder,
    episode: dict,
    device: torch.device,
    graph_microbatch_size: int = 2,
    max_gradient_norm: float | None = 1.0,
) -> EpisodeRunResult:
    """
    Train on one complete few-shot episode.

    Graphs are created and discarded one query at a time.
    """
    model.train()
    optimizer.zero_grad(set_to_none=True)

    n_way = episode["support_patches"].shape[0]
    queries_per_class = episode["query_patches"].shape[1]

    num_queries = n_way * queries_per_class

    logits_rows = []
    targets = []

    total_unscaled_loss = 0.0
    correct = 0

    for query_class_position in range(n_way):
        for query_position in range(queries_per_class):
            query_graphs = build_query_candidate_graphs(
                graph_builder=graph_builder,
                episode=episode,
                query_class_position=query_class_position,
                query_position=query_position,
            )

            # [N]
            query_scores = score_graph_list(
                model=model,
                graphs=query_graphs.graphs,
                device=device,
                graph_microbatch_size=graph_microbatch_size,
            )

            target = torch.tensor(
                [query_graphs.target],
                dtype=torch.long,
                device=device,
            )

            # [1, N] against [1]
            query_loss = F.cross_entropy(
                query_scores.unsqueeze(0),
                target,
            )

            # Accumulated gradients equal the gradient of the
            # mean loss across all query images.
            (
                query_loss / num_queries
            ).backward()

            prediction = int(
                query_scores.argmax().item()
            )

            correct += int(
                prediction == query_graphs.target
            )

            total_unscaled_loss += float(
                query_loss.detach().item()
            )

            logits_rows.append(
                query_scores.detach().cpu()
            )

            targets.append(query_graphs.target)

            # The five graph objects are no longer needed.
            del query_graphs, query_scores, query_loss

    if max_gradient_norm is not None:
        gradient_norm = clip_grad_norm_(
            model.parameters(),
            max_norm=max_gradient_norm,
        )

        gradient_norm_value = float(
            gradient_norm.item()
        )

    else:
        gradient_norm_value = None

    for name, parameter in model.named_parameters():
        if (
            parameter.grad is not None
            and not torch.isfinite(parameter.grad).all()
        ):
            raise RuntimeError(
                f"Non-finite gradient in parameter {name!r}."
            )

    optimizer.step()

    logits = torch.stack(logits_rows)
    targets_tensor = torch.tensor(
        targets,
        dtype=torch.long,
    )

    return EpisodeRunResult(
        loss=total_unscaled_loss / num_queries,
        accuracy=correct / num_queries,
        correct=correct,
        num_queries=num_queries,
        logits=logits,
        targets=targets_tensor,
        gradient_norm=gradient_norm_value,
    )

In [195]:
@torch.inference_mode()
def evaluate_feature_episode(
    model: torch.nn.Module,
    graph_builder: ClassConditionedPatchGraphBuilder,
    episode: dict,
    device: torch.device,
    graph_microbatch_size: int = 2,
) -> EpisodeRunResult:
    model.eval()

    n_way = episode["support_patches"].shape[0]
    queries_per_class = episode["query_patches"].shape[1]

    num_queries = n_way * queries_per_class

    logits_rows = []
    targets = []

    total_loss = 0.0
    correct = 0

    for query_class_position in range(n_way):
        for query_position in range(queries_per_class):
            query_graphs = build_query_candidate_graphs(
                graph_builder=graph_builder,
                episode=episode,
                query_class_position=query_class_position,
                query_position=query_position,
            )

            query_scores = score_graph_list(
                model=model,
                graphs=query_graphs.graphs,
                device=device,
                graph_microbatch_size=graph_microbatch_size,
            )

            target = torch.tensor(
                [query_graphs.target],
                dtype=torch.long,
                device=device,
            )

            loss = F.cross_entropy(
                query_scores.unsqueeze(0),
                target,
            )

            prediction = int(
                query_scores.argmax().item()
            )

            correct += int(
                prediction == query_graphs.target
            )

            total_loss += float(loss.item())

            logits_rows.append(
                query_scores.cpu()
            )

            targets.append(query_graphs.target)

    return EpisodeRunResult(
        loss=total_loss / num_queries,
        accuracy=correct / num_queries,
        correct=correct,
        num_queries=num_queries,
        logits=torch.stack(logits_rows),
        targets=torch.tensor(
            targets,
            dtype=torch.long,
        ),
    )

In [196]:
@dataclass
class SplitMetrics:
    loss: float
    accuracy: float
    correct: int
    num_queries: int
    num_episodes: int


def train_epoch(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    graph_builder: ClassConditionedPatchGraphBuilder,
    episode_dataset,
    device: torch.device,
    epoch: int,
    num_episodes: int,
    graph_microbatch_size: int = 2,
    log_interval: int = 10,
) -> SplitMetrics:
    if hasattr(episode_dataset, "set_epoch"):
        episode_dataset.set_epoch(epoch)

    num_episodes = min(
        num_episodes,
        len(episode_dataset),
    )

    total_loss = 0.0
    total_correct = 0
    total_queries = 0

    for episode_index in range(num_episodes):
        episode = episode_dataset[episode_index]

        result = train_feature_episode(
            model=model,
            optimizer=optimizer,
            graph_builder=graph_builder,
            episode=episode,
            device=device,
            graph_microbatch_size=graph_microbatch_size,
            max_gradient_norm=1.0,
        )

        total_loss += (
            result.loss * result.num_queries
        )

        total_correct += result.correct
        total_queries += result.num_queries

        if (
            log_interval > 0
            and (episode_index + 1) % log_interval == 0
        ):
            print(
                f"  train episode "
                f"{episode_index + 1:4d}/{num_episodes}: "
                f"loss={total_loss / total_queries:.4f}, "
                f"accuracy={total_correct / total_queries:.4f}"
            )

    return SplitMetrics(
        loss=total_loss / total_queries,
        accuracy=total_correct / total_queries,
        correct=total_correct,
        num_queries=total_queries,
        num_episodes=num_episodes,
    )

In [197]:
@torch.inference_mode()
def evaluate_episode_dataset(
    model: torch.nn.Module,
    graph_builder: ClassConditionedPatchGraphBuilder,
    episode_dataset,
    device: torch.device,
    num_episodes: int,
    graph_microbatch_size: int = 2,
    log_interval: int = 10,
) -> SplitMetrics:
    model.eval()

    num_episodes = min(
        num_episodes,
        len(episode_dataset),
    )

    total_loss = 0.0
    total_correct = 0
    total_queries = 0

    for episode_index in range(num_episodes):
        episode = episode_dataset[episode_index]

        result = evaluate_feature_episode(
            model=model,
            graph_builder=graph_builder,
            episode=episode,
            device=device,
            graph_microbatch_size=graph_microbatch_size,
        )

        total_loss += (
            result.loss * result.num_queries
        )

        total_correct += result.correct
        total_queries += result.num_queries

        if (
            log_interval > 0
            and (episode_index + 1) % log_interval == 0
        ):
            print(
                f"  validation episode "
                f"{episode_index + 1:4d}/{num_episodes}: "
                f"loss={total_loss / total_queries:.4f}, "
                f"accuracy={total_correct / total_queries:.4f}"
            )

    return SplitMetrics(
        loss=total_loss / total_queries,
        accuracy=total_correct / total_queries,
        correct=total_correct,
        num_queries=total_queries,
        num_episodes=num_episodes,
    )

In [198]:
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/CrossImageGLOT"
)

CHECKPOINT_DIR = (
    DRIVE_PROJECT_DIR
    / "checkpoints"
    / "graphsage_v1"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_CHECKPOINT = (
    CHECKPOINT_DIR / "latest.pt"
)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR / "best.pt"
)

In [199]:
def save_checkpoint_atomic(
    checkpoint: dict,
    output_path: Path,
) -> None:
    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    torch.save(
        checkpoint,
        temporary_path,
    )

    os.replace(
        temporary_path,
        output_path,
    )

In [200]:
def make_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_validation_accuracy: float,
    epochs_without_improvement: int,
    history: list[dict],
    configuration: dict,
) -> dict:
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_validation_accuracy": (
            best_validation_accuracy
        ),
        "epochs_without_improvement": (
            epochs_without_improvement
        ),
        "history": history,
        "configuration": configuration,
    }

In [201]:
def move_optimizer_state_to_device(
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)

In [202]:
def load_training_checkpoint(
    checkpoint_path: Path,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.to(device)

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    move_optimizer_state_to_device(
        optimizer,
        device,
    )

    return checkpoint

In [203]:
smoke_model = CrossImageGraphMatcher(
    input_dim=387,
    hidden_dim=256,
    num_layers=2,
    dropout=0.1,
    temperature=0.1,
    learnable_temperature=False,
).to(device)

smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

In [206]:
smoke_train_metrics = train_epoch(
    model=smoke_model,
    optimizer=smoke_optimizer,
    graph_builder=graph_builder,
    episode_dataset=train_episodes,
    device=device,
    epoch=0,
    num_episodes=3,
    graph_microbatch_size=2,
    log_interval=1,
)

print("Smoke train:", smoke_train_metrics)

  train episode    1/3: loss=0.5256, accuracy=1.0000
  train episode    2/3: loss=0.5956, accuracy=0.9000
  train episode    3/3: loss=0.4398, accuracy=0.9333
Smoke train: SplitMetrics(loss=0.4397889695440729, accuracy=0.9333333333333333, correct=14, num_queries=15, num_episodes=3)


In [208]:
smoke_val_metrics = evaluate_episode_dataset(
    model=smoke_model,
    graph_builder=graph_builder,
    episode_dataset=val_episodes,
    device=device,
    num_episodes=5,
    graph_microbatch_size=2,
    log_interval=1,
)

print("Smoke validation:", smoke_val_metrics)

  validation episode    1/5: loss=0.2323, accuracy=0.9467
  validation episode    2/5: loss=0.3091, accuracy=0.9333
  validation episode    3/5: loss=0.2892, accuracy=0.9333
  validation episode    4/5: loss=0.2706, accuracy=0.9400
  validation episode    5/5: loss=0.2536, accuracy=0.9520
Smoke validation: SplitMetrics(loss=0.2536269381418824, accuracy=0.952, correct=357, num_queries=375, num_episodes=5)


In [209]:
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F


@dataclass
class BaselineMetrics:
    loss: float
    accuracy: float
    correct: int
    num_queries: int
    num_episodes: int
    episode_accuracy_mean: float
    episode_accuracy_ci95: float

In [210]:
@torch.inference_mode()
def frozen_baseline_episode(
    episode: dict,
    representation: str,
    device: torch.device,
    temperature: float = 0.1,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Return logits and targets for one episode.

    representation:
        "cls"        — use DINOv2 CLS embeddings
        "mean_patch" — average DINOv2 patch embeddings per image
    """
    if representation == "cls":
        support = episode["support_cls"].to(
            device=device,
            dtype=torch.float32,
        )  # [N, K, D]

        query = episode["query_cls"].to(
            device=device,
            dtype=torch.float32,
        )  # [N, Q, D]

    elif representation == "mean_patch":
        support = episode["support_patches"].to(
            device=device,
            dtype=torch.float32,
        ).mean(dim=-2)  # [N, K, D]

        query = episode["query_patches"].to(
            device=device,
            dtype=torch.float32,
        ).mean(dim=-2)  # [N, Q, D]

    else:
        raise ValueError(
            "representation must be 'cls' or 'mean_patch'."
        )

    if temperature <= 0:
        raise ValueError("temperature must be positive.")

    # Mean of the K support-image embeddings for each class.
    prototypes = support.mean(dim=1)  # [N, D]

    prototypes = F.normalize(
        prototypes,
        p=2,
        dim=-1,
    )

    query_embeddings = query.reshape(
        -1,
        query.shape[-1],
    )  # [N*Q, D]

    query_embeddings = F.normalize(
        query_embeddings,
        p=2,
        dim=-1,
    )

    # [N*Q, N]
    logits = (
        query_embeddings @ prototypes.T
    ) / temperature

    targets = torch.as_tensor(
        episode["query_labels"],
        dtype=torch.long,
        device=device,
    ).reshape(-1)

    return logits, targets

In [211]:
@torch.inference_mode()
def evaluate_frozen_baseline(
    episode_dataset,
    representation: str,
    device: torch.device,
    num_episodes: int,
    temperature: float = 0.1,
) -> BaselineMetrics:
    num_episodes = min(
        num_episodes,
        len(episode_dataset),
    )

    total_loss = 0.0
    total_correct = 0
    total_queries = 0
    episode_accuracies = []

    for episode_index in range(num_episodes):
        episode = episode_dataset[episode_index]

        logits, targets = frozen_baseline_episode(
            episode=episode,
            representation=representation,
            device=device,
            temperature=temperature,
        )

        loss = F.cross_entropy(
            logits,
            targets,
        )

        predictions = logits.argmax(dim=-1)

        correct = int(
            (predictions == targets).sum().item()
        )

        query_count = targets.numel()
        episode_accuracy = correct / query_count

        total_loss += loss.item() * query_count
        total_correct += correct
        total_queries += query_count
        episode_accuracies.append(episode_accuracy)

    accuracy_tensor = torch.tensor(
        episode_accuracies,
        dtype=torch.float32,
    )

    mean_episode_accuracy = float(
        accuracy_tensor.mean().item()
    )

    if num_episodes > 1:
        standard_error = float(
            accuracy_tensor.std(unbiased=True).item()
            / math.sqrt(num_episodes)
        )

        confidence_interval = 1.96 * standard_error
    else:
        confidence_interval = float("nan")

    return BaselineMetrics(
        loss=total_loss / total_queries,
        accuracy=total_correct / total_queries,
        correct=total_correct,
        num_queries=total_queries,
        num_episodes=num_episodes,
        episode_accuracy_mean=mean_episode_accuracy,
        episode_accuracy_ci95=confidence_interval,
    )

In [212]:
cls_baseline = evaluate_frozen_baseline(
    episode_dataset=val_episodes,
    representation="cls",
    device=device,
    num_episodes=5,
    temperature=0.1,
)

mean_patch_baseline = evaluate_frozen_baseline(
    episode_dataset=val_episodes,
    representation="mean_patch",
    device=device,
    num_episodes=5,
    temperature=0.1,
)

print("CLS baseline:", cls_baseline)
print("Mean-patch baseline:", mean_patch_baseline)
print("GraphSAGE:", smoke_val_metrics)

CLS baseline: BaselineMetrics(loss=0.11862778961658478, accuracy=0.9786666666666667, correct=367, num_queries=375, num_episodes=5, episode_accuracy_mean=0.9786666631698608, episode_accuracy_ci95=0.015680011698164534)
Mean-patch baseline: BaselineMetrics(loss=0.1589938446879387, accuracy=0.9813333333333333, correct=368, num_queries=375, num_episodes=5, episode_accuracy_mean=0.9813333749771118, episode_accuracy_ci95=0.01568000026940132)
GraphSAGE: SplitMetrics(loss=0.2536269381418824, accuracy=0.952, correct=357, num_queries=375, num_episodes=5)


In [213]:
cls_baseline_100 = evaluate_frozen_baseline(
    episode_dataset=val_episodes,
    representation="cls",
    device=device,
    num_episodes=100,
    temperature=0.1,
)

mean_patch_baseline_100 = evaluate_frozen_baseline(
    episode_dataset=val_episodes,
    representation="mean_patch",
    device=device,
    num_episodes=100,
    temperature=0.1,
)

print("CLS baseline:", cls_baseline_100)
print("Mean-patch baseline:", mean_patch_baseline_100)

CLS baseline: BaselineMetrics(loss=0.1295646960288286, accuracy=0.9805333333333334, correct=7354, num_queries=7500, num_episodes=100, episode_accuracy_mean=0.9805333018302917, episode_accuracy_ci95=0.0037036431059241292)
Mean-patch baseline: BaselineMetrics(loss=0.190166439935565, accuracy=0.9636, correct=7227, num_queries=7500, num_episodes=100, episode_accuracy_mean=0.9636000990867615, episode_accuracy_ci95=0.005999178998172283)


In [214]:
REAL_CONFIG = {
    "num_epochs": 10,
    "train_episodes_per_epoch": 100,
    "validation_episodes_per_epoch": 20,
    "final_validation_episodes": 100,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "graph_microbatch_size": 2,
    "early_stopping_patience": 3,
    "input_dim": 387,
    "hidden_dim": 256,
    "num_layers": 2,
    "dropout": 0.1,
    "temperature": 0.1,
}

In [215]:
model = CrossImageGraphMatcher(
    input_dim=REAL_CONFIG["input_dim"],
    hidden_dim=REAL_CONFIG["hidden_dim"],
    num_layers=REAL_CONFIG["num_layers"],
    dropout=REAL_CONFIG["dropout"],
    temperature=REAL_CONFIG["temperature"],
    learnable_temperature=False,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=REAL_CONFIG["learning_rate"],
    weight_decay=REAL_CONFIG["weight_decay"],
)

In [216]:
history = []

start_epoch = 0
best_validation_accuracy = float("-inf")
epochs_without_improvement = 0

RESUME_TRAINING = True

if RESUME_TRAINING and LATEST_CHECKPOINT.exists():
    checkpoint = load_training_checkpoint(
        checkpoint_path=LATEST_CHECKPOINT,
        model=model,
        optimizer=optimizer,
        device=device,
    )

    start_epoch = checkpoint["epoch"] + 1

    best_validation_accuracy = checkpoint[
        "best_validation_accuracy"
    ]

    epochs_without_improvement = checkpoint[
        "epochs_without_improvement"
    ]

    history = checkpoint.get("history", [])

    print(
        f"Resumed from epoch {checkpoint['epoch']}."
    )
else:
    print("Starting training from scratch.")

Starting training from scratch.


In [217]:
for epoch in range(
    start_epoch,
    REAL_CONFIG["num_epochs"],
):
    print(f"\nEpoch {epoch + 1}/{REAL_CONFIG['num_epochs']}")

    train_metrics = train_epoch(
        model=model,
        optimizer=optimizer,
        graph_builder=graph_builder,
        episode_dataset=train_episodes,
        device=device,
        epoch=epoch,
        num_episodes=(
            REAL_CONFIG["train_episodes_per_epoch"]
        ),
        graph_microbatch_size=(
            REAL_CONFIG["graph_microbatch_size"]
        ),
        log_interval=10,
    )

    validation_metrics = evaluate_episode_dataset(
        model=model,
        graph_builder=graph_builder,
        episode_dataset=val_episodes,
        device=device,
        num_episodes=(
            REAL_CONFIG["validation_episodes_per_epoch"]
        ),
        graph_microbatch_size=(
            REAL_CONFIG["graph_microbatch_size"]
        ),
        log_interval=5,
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "validation_loss": validation_metrics.loss,
        "validation_accuracy": (
            validation_metrics.accuracy
        ),
    }

    history.append(epoch_record)

    improved = (
        validation_metrics.accuracy
        > best_validation_accuracy
    )

    if improved:
        best_validation_accuracy = (
            validation_metrics.accuracy
        )

        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    checkpoint = make_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        best_validation_accuracy=(
            best_validation_accuracy
        ),
        epochs_without_improvement=(
            epochs_without_improvement
        ),
        history=history,
        configuration=REAL_CONFIG,
    )

    save_checkpoint_atomic(
        checkpoint=checkpoint,
        output_path=LATEST_CHECKPOINT,
    )

    if improved:
        save_checkpoint_atomic(
            checkpoint=checkpoint,
            output_path=BEST_CHECKPOINT,
        )

        print("Saved new best checkpoint.")

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_metrics.loss:.4f}, "
        f"train accuracy={train_metrics.accuracy:.4f}, "
        f"validation loss={validation_metrics.loss:.4f}, "
        f"validation accuracy={validation_metrics.accuracy:.4f}, "
        f"best={best_validation_accuracy:.4f}"
    )

    if (
        epochs_without_improvement
        >= REAL_CONFIG["early_stopping_patience"]
    ):
        print(
            "Early stopping: validation accuracy "
            "did not improve."
        )
        break


Epoch 1/10
  train episode   10/100: loss=0.4713, accuracy=0.8800
  train episode   20/100: loss=0.3390, accuracy=0.9000
  train episode   30/100: loss=0.3004, accuracy=0.9200
  train episode   40/100: loss=0.2804, accuracy=0.9350
  train episode   50/100: loss=0.2389, accuracy=0.9480
  train episode   60/100: loss=0.2324, accuracy=0.9500
  train episode   70/100: loss=0.2108, accuracy=0.9571
  train episode   80/100: loss=0.1955, accuracy=0.9575
  train episode   90/100: loss=0.1881, accuracy=0.9556
  train episode  100/100: loss=0.1950, accuracy=0.9500
  validation episode    5/20: loss=0.1904, accuracy=0.9467
  validation episode   10/20: loss=0.1731, accuracy=0.9520
  validation episode   15/20: loss=0.1963, accuracy=0.9413
  validation episode   20/20: loss=0.2058, accuracy=0.9340
Saved new best checkpoint.
Epoch 1: train loss=0.1950, train accuracy=0.9500, validation loss=0.2058, validation accuracy=0.9340, best=0.9340

Epoch 2/10
  train episode   10/100: loss=0.1013, accuracy=

In [218]:
best_checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

best_validation_metrics = evaluate_episode_dataset(
    model=model,
    graph_builder=graph_builder,
    episode_dataset=val_episodes,
    device=device,
    num_episodes=(
        REAL_CONFIG["final_validation_episodes"]
    ),
    graph_microbatch_size=(
        REAL_CONFIG["graph_microbatch_size"]
    ),
    log_interval=10,
)

print("Best GraphSAGE validation:", best_validation_metrics)
print("Frozen CLS accuracy:", 0.9805333333333334)

  validation episode   10/100: loss=0.1660, accuracy=0.9507
  validation episode   20/100: loss=0.1836, accuracy=0.9427
  validation episode   30/100: loss=0.2008, accuracy=0.9373
  validation episode   40/100: loss=0.2043, accuracy=0.9377
  validation episode   50/100: loss=0.1929, accuracy=0.9421
  validation episode   60/100: loss=0.1862, accuracy=0.9447
  validation episode   70/100: loss=0.1818, accuracy=0.9461
  validation episode   80/100: loss=0.1887, accuracy=0.9430
  validation episode   90/100: loss=0.1879, accuracy=0.9433
  validation episode  100/100: loss=0.1878, accuracy=0.9425
Best GraphSAGE validation: SplitMetrics(loss=0.18782305619895537, accuracy=0.9425333333333333, correct=7069, num_queries=7500, num_episodes=100)
Frozen CLS accuracy: 0.9805333333333334
